<a href="https://colab.research.google.com/github/vishweshwari63/AI_agent_for_SmartFramingAdvice/blob/main/StyleSphere_AI_%E2%80%93_Multi_Agent_Fashion_Stylist.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Install & Imports

In [ ]:
!pip install -q google-generativeai gradio pillow requests

import os
import json
import random
import time
from datetime import datetime
from dataclasses import dataclass, field
from typing import List, Dict, Optional

import requests
from io import BytesIO
from PIL import Image

import google.generativeai as genai
import gradio as gr


Gemini configuration

In [ ]:
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", None)

if GEMINI_API_KEY:
    genai.configure(api_key=GEMINI_API_KEY)
    print("✅ Gemini configured from environment variable.")
else:
    print("⚠️ No GEMINI_API_KEY found. StylistAgent will use a rule-based fallback.")

# Simple logging helper
def log_event(agent_name: str, event: str, data: Optional[dict] = None):
    payload = {"agent": agent_name, "event": event, "data": data or {}}
    print(f"[LOG] {json.dumps(payload)}")


⚠️ No GEMINI_API_KEY found. StylistAgent will use a rule-based fallback.


Product catalog

In [ ]:
# Execute this cell to define the Product class and related functions
from dataclasses import dataclass, field
from typing import List, Dict, Optional
@dataclass
class Product:
    id: str
    name: str
    brand: str
    price: float
    image_url: str
    colors: List[str]
    categories: List[str]

# Use placeholder images that always load
PRODUCTS = [
    Product(
        id="1",
        name="Pastel Linen Shirt",
        brand="Breeze",
        price=40.0,
        image_url="https://tse4.mm.bing.net/th/id/OIP.IEImHxJ4esScQ5pn9wCgGgHaLF?cb=ucfimg2ucfimg=1&w=1400&h=2096&rs=1&pid=ImgDetMain&o=7&rm=3",
        colors=["pastel-pink", "white"],
        categories=["top", "smart-casual"],
    ),
    Product(
        id="2",
        name="White Chino Pants",
        brand="UrbanEase",
        price=45.0,
        image_url="https://via.placeholder.com/400x400.png?text=White+Chinos",
        colors=["white"],
        categories=["bottom", "smart-casual"],
    ),
    Product(
        id="3",
        name="Tan Loafers",
        brand="StepRight",
        price=50.0,
        image_url="https://via.placeholder.com/400x400.png?text=Tan+Loafers",
        colors=["tan"],
        categories=["shoes", "smart-casual"],
    ),
    Product(
        id="4",
        name="Pastel Maxi Dress",
        brand="Sunset Bloom",
        price=70.0,
        image_url="https://via.placeholder.com/400x400.png?text=Pastel+Maxi+Dress",
        colors=["pastel-blue", "pastel-pink"],
        categories=["dress", "occasion", "wedding"],
    ),
    Product(
        id="5",
        name="Strappy Sandals",
        brand="SoleWave",
        price=35.0,
        image_url="https://via.placeholder.com/400x400.png?text=Strappy+Sandals",
        colors=["beige"],
        categories=["shoes", "occasion"],
    ),
    Product(
        id="6",
        name="Statement Pearl Earrings",
        brand="Aurora",
        price=20.0,
        image_url="https://via.placeholder.com/400x400.png?text=Pearl+Earrings",
        colors=["white"],
        categories=["accessory", "occasion"],
    ),
]

print(f"Loaded {len(PRODUCTS)} demo products.")

def product_by_id(pid: str) -> Optional[Product]:
    for p in PRODUCTS:
        if p.id == pid:
            return p
    return None

def product_search(
    max_price: Optional[float] = None,
    min_price: Optional[float] = None,
    colors: Optional[List[str]] = None,
    categories: Optional[List[str]] = None,
    limit: int = 10,
) -> List[Product]:
    results = []
    for p in PRODUCTS:
        if max_price is not None and p.price > max_price:
            continue
        if min_price is not None and p.price < min_price:
            continue
        if colors and not any(c in p.colors for c in colors):
            continue
        if categories and not any(cat in p.categories for cat in categories):
            continue
        results.append(p)
    log_event("product_search", "filtered", {"count": len(results)})
    return results[:limit]

def color_harmony_score(items: List[Product]) -> float:
    if not items:
        return 0.0
    score = 0
    for p in items:
        for c in p.colors:
            if "pastel" in c or c in ["white", "beige", "tan"]:
                score += 2
            else:
                score += 1
    return min(1.0, score / (len(items) * 3))

Loaded 6 demo products.


Sessions & memory

In [ ]:
@dataclass
class UserPreferences:
    budget: Optional[float] = None
    styles: List[str] = field(default_factory=list)
    disliked_colors: List[str] = field(default_factory=list)
    preferred_fit: Optional[str] = None

@dataclass
class WardrobeItem:
    id: str
    name: str
    colors: List[str]
    category: str
    notes: str = ""

@dataclass
class SessionState:
    user_id: str
    preferences: UserPreferences = field(default_factory=UserPreferences)
    wardrobe: List[WardrobeItem] = field(default_factory=list)
    last_outfits: List[dict] = field(default_factory=list)

class SessionService:
    def __init__(self):
        self.sessions: Dict[str, SessionState] = {}

    def get_session(self, user_id: str) -> SessionState:
        if user_id not in self.sessions:
            self.sessions[user_id] = SessionState(user_id=user_id)
            log_event("SessionService", "create_session", {"user_id": user_id})
        return self.sessions[user_id]

    def update_preferences(self, user_id: str, **kwargs):
        s = self.get_session(user_id)
        for k, v in kwargs.items():
            setattr(s.preferences, k, v)
        log_event("SessionService", "update_preferences", {"user_id": user_id, "updates": kwargs})

session_service = SessionService()
DEMO_USER_ID = "demo_user"


Agents:Stylist,Budget,Closet,Trend

In [ ]:
class StylistAgent:
    """Uses Gemini if available; otherwise falls back to simple rules."""
    def __init__(self):
        self.model = genai.GenerativeModel("gemini-1.5-pro") if GEMINI_API_KEY else None

    def create_outfit(self, user_message: str, session: SessionState, candidate_products: List[Product]) -> dict:
        log_event("StylistAgent", "create_outfit_start", {"msg": user_message})
        if self.model is None:
            return self._fallback_outfit(user_message, session, candidate_products)

        products_json = [
            {
                "id": p.id,
                "name": p.name,
                "brand": p.brand,
                "price": p.price,
                "colors": p.colors,
                "categories": p.categories,
            }
            for p in candidate_products
        ]

        prompt = f"""
You are a professional fashion stylist AI.

User message:
\"\"\"{user_message}\"\"\"

User preferences:
- Budget: {session.preferences.budget}
- Styles: {", ".join(session.preferences.styles) if session.preferences.styles else "unknown"}
- Disliked colors: {", ".join(session.preferences.disliked_colors) if session.preferences.disliked_colors else "none"}

Catalog products (JSON):
{json.dumps(products_json[:25], indent=2)}

Select 3–5 items that form a cohesive outfit for the user's request.
Return ONLY valid JSON in this structure:

{{
  "name": "short outfit title",
  "description": "1–3 sentence description of the outfit and why it works",
  "item_ids": ["1", "2", "3"],
  "style_tags": ["tag1", "tag2"],
  "estimated_price": 123.45
}}
"""

        try:
            resp = self.model.generate_content(prompt)
            text = resp.text
            data = None
            try:
                data = json.loads(text)
            except Exception:
                start = text.find("{")
                end = text.rfind("}")
                if start != -1 and end != -1:
                    data = json.loads(text[start:end+1])
            if not data:
                log_event("StylistAgent", "parse_error_fallback", {})
                return self._fallback_outfit(user_message, session, candidate_products)
            log_event("StylistAgent", "create_outfit_success", {"item_ids": data.get("item_ids", [])})
            return data
        except Exception as e:
            log_event("StylistAgent", "error_fallback", {"error": str(e)})
            return self._fallback_outfit(user_message, session, candidate_products)

    def _fallback_outfit(self, user_message: str, session: SessionState, candidate_products: List[Product]) -> dict:
        items = []
        lower = user_message.lower()
        if "wedding" in lower:
            dresses = [p for p in candidate_products if "dress" in p.categories]
            shoes = [p for p in candidate_products if "shoes" in p.categories]
            accessories = [p for p in candidate_products if "accessory" in p.categories]
            if dresses:
                items.append(random.choice(dresses).id)
            if shoes:
                items.append(random.choice(shoes).id)
            if accessories:
                items.append(random.choice(accessories).id)
        else:
            for p in candidate_products[:3]:
                items.append(p.id)
        total_price = sum(product_by_id(pid).price for pid in items)
        return {
            "name": "Simple Styled Look",
            "description": "A simple fallback outfit using matching items from the catalog.",
            "item_ids": items,
            "style_tags": ["fallback"],
            "estimated_price": total_price,
        }

class BudgetAgent:
    def optimize_outfit(self, outfit: dict, session: SessionState) -> dict:
        budget = session.preferences.budget
        if budget is None:
            return outfit
        ids = outfit["item_ids"]
        items = [product_by_id(pid) for pid in ids if product_by_id(pid)]
        total = sum(p.price for p in items)
        if total <= budget:
            outfit["budget_status"] = f"✅ Within budget (${total:.2f} of ${budget:.2f})"
            return outfit

        items_sorted = sorted(items, key=lambda p: p.price, reverse=True)
        for expensive in items_sorted:
            candidates = product_search(
                max_price=expensive.price - 10,
                categories=expensive.categories,
                limit=3,
            )
            if not candidates:
                continue
            cheaper = candidates[0]
            outfit["item_ids"].remove(expensive.id)
            outfit["item_ids"].append(cheaper.id)
            break

        new_items = [product_by_id(pid) for pid in outfit["item_ids"] if product_by_id(pid)]
        total_new = sum(p.price for p in new_items)
        status = "✅" if total_new <= budget else "⚠️"
        outfit["estimated_price"] = total_new
        outfit["budget_status"] = f"{status} Adjusted price: ${total_new:.2f} (budget ${budget:.2f})"
        return outfit

class ClosetAgent:
    def add_item(self, session: SessionState, name: str, colors: List[str], category: str, notes: str = ""):
        wid = f"w{len(session.wardrobe)+1}"
        item = WardrobeItem(id=wid, name=name, colors=colors, category=category, notes=notes)
        session.wardrobe.append(item)
        log_event("ClosetAgent", "add_item", {"id": wid})
        return item

    def list_items(self, session: SessionState) -> List[WardrobeItem]:
        return session.wardrobe

class TrendAgent:
    def __init__(self):
        self.trends = []
        self.last_refreshed = None

    def refresh_trends(self):
        log_event("TrendAgent", "refresh_start", {})
        time.sleep(1.0)
        now = datetime.utcnow().isoformat() + "Z"
        self.last_refreshed = now
        self.trends = [
            {
                "name": "Soft Pastel Wedding Guest",
                "vibe": "Light, romantic, flowy silhouettes in pastel tones.",
                "tags": ["pastel", "wedding", "romantic"],
            },
            {
                "name": "Minimal Resort Linen",
                "vibe": "Crisp whites and linens, relaxed tailoring, beach-perfect.",
                "tags": ["minimal", "linen", "resort"],
            },
            {
                "name": "Statement Accessories",
                "vibe": "Clean base outfits with bold earrings and bags.",
                "tags": ["accessories", "statement", "elevated-basics"],
            },
        ]
        log_event("TrendAgent", "refresh_done", {"count": len(self.trends)})
        return self.trends, now

    def get_trends(self):
        if not self.trends:
            trends, _ = self.refresh_trends()
            return trends
        return self.trends

trend_agent = TrendAgent()



Evaluation & visualization helpers

In [ ]:
def evaluate_outfit(outfit: dict, session: SessionState) -> dict:
    budget = session.preferences.budget or 200.0
    est = outfit.get("estimated_price", 0.0)
    if est <= budget:
        budget_fit = 1.0
    else:
        over_ratio = (est - budget) / max(budget, 1.0)
        budget_fit = max(0.0, 1.0 - over_ratio)
    items = [product_by_id(pid) for pid in outfit["item_ids"] if product_by_id(pid)]
    color_score = color_harmony_score(items)
    count = len(outfit["item_ids"])
    item_score = 1.0 if 3 <= count <= 5 else max(0.0, 1.0 - abs(count - 4)*0.25)
    overall = 0.4*budget_fit + 0.4*color_score + 0.2*item_score
    return {
        "budget_fit": round(budget_fit, 2),
        "color_harmony": round(color_score, 2),
        "item_count_score": round(item_score, 2),
        "overall": round(overall, 2),
    }

COLOR_MAP = {
    "pastel-pink": "#ffc1cc",
    "pastel-blue": "#c4e1ff",
    "white": "#ffffff",
    "beige": "#f5f5dc",
    "tan": "#d2b48c",
    "default": "#cccccc",
}

def color_to_hex(c: str) -> str:
    return COLOR_MAP.get(c.lower(), COLOR_MAP["default"])

def build_palette_html(items: list) -> str:
    colors = []
    for p in items:
        for c in p.get("colors", []):
            if c not in colors:
                colors.append(c)
    if not colors:
        return ""
    swatches = []
    for c in colors:
        swatches.append(
            f"<div style='display:inline-block;width:22px;height:22px;border-radius:999px;"
            f"margin-right:6px;background:{color_to_hex(c)};border:1px solid #e5e7eb;' title='{c}'></div>"
        )
    return "<div><strong>Palette:</strong><br>" + "".join(swatches) + "</div>"

def highlight_trends_for_outfit(outfit: dict) -> str:
    trends = trend_agent.get_trends()
    tags = set(outfit.get("style_tags", []))
    if not tags:
        return ""
    matches = []
    for t in trends:
        if tags.intersection(set(t["tags"])):
            matches.append(t["name"])
    if not matches:
        return ""
    lines = ["🔥 **Trend highlights:**"] + [f"- {m}" for m in matches]
    return "\n".join(lines)

def build_outfit_card_html(outfit: dict, items: list) -> str:
    name = outfit.get("name", "Styled Look")
    desc = outfit.get("description", "")
    scores = outfit.get("scores", {})
    est_price = outfit.get("estimated_price", 0.0)
    budget_status = outfit.get("budget_status", "")
    overall = scores.get("overall", None)
    color_score = scores.get("color_harmony", "?")
    score_str = f"<span style='font-weight:600;'>Overall:</span> {overall}" if overall is not None else ""
    return f"""
<div style="
    background: #ffffff;
    border-radius: 18px;
    padding: 12px 16px;
    border: 1px solid #f3e8ff;
    box-shadow: 0 10px 25px rgba(15, 23, 42, 0.06);
    color: #0f172a;
    font-family: system-ui, -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif;
">
  <div style="font-size: 16px; font-weight: 650; margin-bottom: 4px; color:#4b164c;">
    👗 {name}
  </div>
  <div style="font-size: 12px; opacity: 0.9; margin-bottom: 8px;">
    {desc}
  </div>
  <div style="font-size: 12px; margin-bottom: 2px;">
    <span style="font-weight:600;">Total:</span> ${est_price:.2f}
  </div>
  <div style="font-size: 11px; margin-bottom: 2px; color:#16a34a;">
    {budget_status}
  </div>
  <div style="font-size: 11px; opacity: 0.9;">
    {score_str} &nbsp; <span style="font-weight:600;">Color harmony:</span> {color_score}
  </div>
</div>
"""

def build_outfit_board_image(items: list):
    if not items:
        return None
    try:
        imgs = []
        target_h = 220
        for p in items:
            url = p.get("image_url")
            if not url:
                continue
            resp = requests.get(url, timeout=5)
            img = Image.open(BytesIO(resp.content)).convert("RGB")
            w, h = img.size
            new_w = int(w * (target_h / h))
            imgs.append(img.resize((new_w, target_h)))
        if not imgs:
            return None
        total_w = sum(img.size[0] for img in imgs)
        board = Image.new("RGB", (total_w, target_h), (250, 250, 255))
        x = 0
        for img in imgs:
            board.paste(img, (x, 0))
            x += img.size[0]
        return board
    except Exception as e:
        log_event("UI", "board_image_error", {"error": str(e)})
        return None

def build_products_html(items: list) -> str:
    if not items:
        return "<div style='font-size:13px; opacity:0.7;'>No products in this look yet.</div>"
    cards = []
    for p in items:
        img_url = p.get("image_url", "")
        name = p.get("name", "Item")
        brand = p.get("brand", "")
        price = p.get("price", 0)
        colors = ", ".join(p.get("colors", []))
        cards.append(f"""
        <div style="
            min-width: 180px;
            max-width: 200px;
            background:#ffffff;
            border-radius:14px;
            border:1px solid #f3e8ff;
            box-shadow:0 6px 16px rgba(15,23,42,0.06);
            padding:10px;
            display:flex;
            flex-direction:column;
            gap:6px;
        ">
          <div style="width:100%;border-radius:10px;overflow:hidden;background:#f9fafb;">
            <img src="{img_url}" alt="{name}" style="width:100%;display:block;object-fit:cover;" />
          </div>
          <div style="font-size:12px;font-weight:600;line-height:1.2;">
            {name}
          </div>
          <div style="font-size:11px;opacity:0.7;">
            {brand}
          </div>
          <div style="font-size:12px;font-weight:600;color:#4b164c;">
            ${price:.2f}
          </div>
          <div style="font-size:11px;opacity:0.7;">
            {colors}
          </div>
        </div>
        """)
    return f"""
<div style="overflow-x:auto;padding:4px 0 8px 0;">
  <div style="display:flex;flex-wrap:nowrap;gap:12px;">
    {''.join(cards)}
  </div>
</div>
"""

def format_outfit_markdown(outfit: dict, items: list) -> str:
    lines = [f"**{outfit.get('name', 'Outfit')}**"]
    if outfit.get("description"):
        lines.append(outfit["description"])
    if outfit.get("estimated_price") is not None:
        lines.append(f"Total: `${outfit['estimated_price']:.2f}`")
    return "\n\n".join(lines)



Orchestrator agent

In [ ]:
class OrchestratorAgent:
    def __init__(self):
        self.stylist = StylistAgent()
        self.budget = BudgetAgent()
        self.closet = ClosetAgent()

    def classify_intent(self, message: str) -> str:
        msg = message.lower()
        if "show closet" in msg or "my wardrobe" in msg:
            return "SHOW_CLOSET"
        if "add to closet" in msg or "save this item" in msg:
            return "ADD_TO_CLOSET"
        return "OUTFIT_REQUEST"

    def handle_message(self, user_id: str, message: str) -> dict:
        session = session_service.get_session(user_id)
        intent = self.classify_intent(message)
        log_event("Orchestrator", "intent", {"intent": intent})

        if intent == "OUTFIT_REQUEST":
            return self._handle_outfit(session, message)
        elif intent == "SHOW_CLOSET":
            items = self.closet.list_items(session)
            return {
                "type": "closet",
                "text": f"You have {len(items)} items in your closet.",
                "items": [w.__dict__ for w in items],
            }
        elif intent == "ADD_TO_CLOSET":
            item = self.closet.add_item(session, "User's white shirt", ["white"], "top", "Added from chat")
            return {"type": "closet_add", "text": f"Added {item.name} to your closet."}
        else:
            return {"type": "info", "text": "I can help you style outfits and manage your closet."}

    def _handle_outfit(self, session: SessionState, message: str) -> dict:
        words = message.lower().replace("$", "").split()
        nums = [w for w in words if w.replace(".", "", 1).isdigit()]
        if nums:
            session.preferences.budget = float(nums[0])
        candidates = product_search(max_price=session.preferences.budget, limit=20) if session.preferences.budget else PRODUCTS
        outfit = self.stylist.create_outfit(message, session, candidates)
        outfit = self.budget.optimize_outfit(outfit, session)
        items = [product_by_id(pid).__dict__ for pid in outfit["item_ids"] if product_by_id(pid)]
        outfit["scores"] = evaluate_outfit(outfit, session)
        session.last_outfits.append(outfit)
        return {"type": "outfit", "outfit": outfit, "items": items}

orchestrator = OrchestratorAgent()


Trend & closet helpers + virtual try-on

In [ ]:
def load_trends_ui():
    trends = trend_agent.get_trends()
    ts = trend_agent.last_refreshed
    info = f"Last refreshed at: {ts}" if ts else "Trends not refreshed yet."
    table = [[t["name"], t["vibe"], ", ".join(t["tags"])] for t in trends]
    return info, table

def refresh_trends_ui():
    trends, ts = trend_agent.refresh_trends()
    info = f"Last refreshed at: {ts}"
    table = [[t["name"], t["vibe"], ", ".join(t["tags"])] for t in trends]
    return info, table

def add_closet_item_ui(name, colors_text, category, notes):
    session = session_service.get_session(DEMO_USER_ID)
    if not name:
        return "Please provide a name.", show_closet()
    colors = [c.strip() for c in (colors_text or "").split(",") if c.strip()]
    item = orchestrator.closet.add_item(session, name, colors, category, notes)
    return f"Added '{item.name}' to your closet.", show_closet()

def show_closet():
    session = session_service.get_session(DEMO_USER_ID)
    items = session.wardrobe
    return [[w.id, w.name, ", ".join(w.colors), w.category, w.notes] for w in items]

def get_last_outfit_board(user_id: str = DEMO_USER_ID):
    session = session_service.get_session(user_id)
    if not session.last_outfits:
        return None, "No outfit yet. Ask the stylist first."
    last_outfit = session.last_outfits[-1]
    items = [product_by_id(pid).__dict__ for pid in last_outfit["item_ids"] if product_by_id(pid)]
    board = build_outfit_board_image(items)
    if board is None:
        return None, "Could not build outfit board."
    return board, f"Using your last outfit: {last_outfit.get('name', 'Styled Look')}"

def virtual_try_on(user_photo):
    if user_photo is None:
        return None, None, "Upload a photo first."
    board, info = get_last_outfit_board()
    return user_photo, board, info



style_chat backend for Gradio

In [ ]:
def style_chat(message, chat_history, budget_slider, style_tags, event_select, tone_select):
    if chat_history is None:
        chat_history = []

    session = session_service.get_session(DEMO_USER_ID)
    if style_tags:
        session.preferences.styles = list(style_tags)
    if budget_slider and budget_slider > 0:
        session.preferences.budget = float(budget_slider)

    if event_select and event_select != "Auto-detect":
        message = f"{message} (Event: {event_select})"
    if tone_select and tone_select != "Any":
        message = f"{message} (Color tone: {tone_select})"

    result = orchestrator.handle_message(DEMO_USER_ID, message)

    reply_md = ""
    outfit_card_html = ""
    palette_html = ""
    trend_md = ""
    items_table = []
    products_html = ""
    board_img = None

    if result["type"] == "outfit":
        outfit = result["outfit"]
        items = result["items"]
        reply_md = format_outfit_markdown(outfit, items)
        outfit_card_html = build_outfit_card_html(outfit, items)
        palette_html = build_palette_html(items)
        trend_md = highlight_trends_for_outfit(outfit)
        items_table = [
            [p["id"], p["name"], p["brand"], p["price"], ", ".join(p["colors"]), ", ".join(p["categories"])]
            for p in items
        ]
        products_html = build_products_html(items)
        board_img = build_outfit_board_image(items)
    elif result["type"] == "closet":
        reply_md = result["text"]
    else:
        reply_md = result.get("text", "I can help you build outfits!")

    chat_history = chat_history + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": reply_md},
    ]

    return (
        chat_history,     # chatbot
        reply_md,         # outfit_md
        outfit_card_html, # outfit_card
        palette_html,     # palette_bar
        trend_md,         # trend_md
        items_table,      # items_table
        products_html,    # products_html_comp
        board_img,        # board_image
    )



Gradio UI (boutique, chat on top, product rail)

In [ ]:
theme = gr.themes.Soft(
    primary_hue="pink",
    secondary_hue="indigo",
    radius_size="lg",
).set(
    body_background_fill="#faf5ff",
    body_text_color="#111827",
    block_background_fill="#ffffff",
    block_border_width="1px",
    block_border_color="#f3e8ff",
)

with gr.Blocks(title="StyleSphere AI – Personal Fashion Assistant", theme=theme) as demo:
    gr.Markdown(
        """
# 🌸 StyleSphere AI

A mini **boutique-style** personal fashion assistant:
Curated outfits · Your own closet · Trend-aware looks
"""
    )

    # ---------- TAB 1: AI Stylist ----------
    with gr.Tab("AI Stylist"):
        gr.Markdown("### 💬 Talk to your stylist")

        with gr.Row():
            with gr.Column(scale=3):
                chatbot = gr.Chatbot(
                    label="AI Stylist Chat",
                    height=320,
                    type="messages",
                )
            with gr.Column(scale=2):
                user_input = gr.Textbox(
                    label="Tell me what you need",
                    placeholder="e.g. I need an outfit for a beach wedding under $150. I like pastel colors.",
                )
                event_select = gr.Dropdown(
                    ["Auto-detect", "Wedding", "Office", "Casual", "Date", "Travel"],
                    value="Auto-detect",
                    label="Event",
                )
                style_tags = gr.CheckboxGroup(
                    ["pastel", "minimal", "streetwear", "formal", "boho"],
                    label="Style vibe",
                )
                budget_slider = gr.Slider(
                    minimum=0,
                    maximum=300,
                    value=0,
                    step=10,
                    label="Max budget (0 = no limit)",
                )
                tone_select = gr.Dropdown(
                    ["Any", "Light/Pastel", "Neutral", "Dark/Bold"],
                    value="Any",
                    label="Color tone",
                )
                send_btn = gr.Button("Style me ✨", variant="primary")

        gr.Markdown("### ⭐ Featured outfit")

        with gr.Row():
            with gr.Column(scale=2):
                outfit_card = gr.HTML(label="Outfit Card")
                palette_bar = gr.HTML(label="Color Palette")
            with gr.Column(scale=2):
                outfit_md = gr.Markdown(label="Quick notes")
                trend_md = gr.Markdown(label="Trend highlights")

        gr.Markdown("### 🛍️ Shop this look")

        products_html_comp = gr.HTML(label="Products")

        board_image = gr.Image(
            label="Outfit collage (preview)",
            interactive=False,
            height=220,
        )

        def on_send(message, chat_history, budget_value, styles_value, event_value, tone_value):
            return style_chat(message, chat_history, budget_value, styles_value, event_value, tone_value)

        send_btn.click(
            on_send,
            inputs=[user_input, chatbot, budget_slider, style_tags, event_select, tone_select],
            outputs=[
                chatbot,
                outfit_md,
                outfit_card,
                palette_bar,
                trend_md,
                products_html_comp,
                board_image,
            ],
        )
        user_input.submit(
            on_send,
            inputs=[user_input, chatbot, budget_slider, style_tags, event_select, tone_select],
            outputs=[
                chatbot,
                outfit_md,
                outfit_card,
                palette_bar,
                trend_md,
                products_html_comp,
                board_image,
            ],
        )

    # ---------- TAB 2: My Closet ----------
    with gr.Tab("My Closet"):
        gr.Markdown("### 👚 Your Closet")

        with gr.Row():
            with gr.Column():
                closet_name = gr.Textbox(label="Item name", placeholder="e.g. White linen shirt")
                closet_colors = gr.Textbox(label="Colors (comma-separated)", placeholder="white, beige")
                closet_category = gr.Dropdown(
                    ["top", "bottom", "dress", "shoes", "accessory", "outerwear"],
                    label="Category",
                    value="top",
                )
                closet_notes = gr.Textbox(label="Notes (optional)")
                add_btn = gr.Button("Add to Closet")
                add_msg = gr.Markdown(label="Status")
            with gr.Column():
                closet_grid = gr.HTML(label="Closet items")

        demo.load(show_closet_html, inputs=None, outputs=closet_grid)

        add_btn.click(
            add_closet_item_ui,
            inputs=[closet_name, closet_colors, closet_category, closet_notes],
            outputs=[add_msg, closet_grid],
        )

    # ---------- TAB 3: Trends ----------
    with gr.Tab("Trends"):
        gr.Markdown("### 🔮 Fashion Trends")

        trends_info = gr.Markdown(label="Refresh Info")
        trends_grid = gr.HTML(label="Trend grid")
        refresh_btn = gr.Button("Refresh Trends ♻️")

        demo.load(load_trends_ui, inputs=None, outputs=[trends_info, trends_grid])
        refresh_btn.click(
            refresh_trends_ui,
            inputs=None,
            outputs=[trends_info, trends_grid],
        )

    # ---------- TAB 4: Virtual Try-On ----------
    with gr.Tab("Virtual Try-On"):
        gr.Markdown(
            "### 🪞 Virtual Try-On (Preview)\n"
            "_Upload your photo and see it next to your latest styled outfit._"
        )

        with gr.Row():
            with gr.Column():
                user_photo = gr.Image(label="Upload your photo", type="pil")
                try_btn = gr.Button("Try my last outfit ✨")
                vt_caption = gr.Markdown(label="Info")
            with gr.Column():
                vt_user_preview = gr.Image(label="Your photo", interactive=False)
                vt_outfit_board = gr.Image(label="Outfit board", interactive=False)

        try_btn.click(
            virtual_try_on,
            inputs=[user_photo],
            outputs=[vt_user_preview, vt_outfit_board, vt_caption],
        )

demo.launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5fef4fbd823880b2a5.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


⚠️ No GEMINI_API_KEY found. StylistAgent will use a rule-based fallback.
Loaded 6 demo products.
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4653e30486bf7becce.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# ============================================================
# StyleSphere AI – Boutique Multi-Agent Fashion Assistant
# (No spreadsheet UI – only cards & grids, with more products)
# ============================================================

# ---------- Cell 1: Install & imports ----------
!pip install -q google-generativeai gradio pillow requests

import os
import json
import random
import time
from datetime import datetime
from dataclasses import dataclass, field
from typing import List, Dict, Optional

import requests
from io import BytesIO
from PIL import Image  # still used for outfit collage

import google.generativeai as genai
import gradio as gr

# ---------- Cell 2: Gemini config ----------
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", None)

if GEMINI_API_KEY:
    genai.configure(api_key=GEMINI_API_KEY)
    print("✅ Gemini configured from environment variable.")
else:
    print("⚠️ No GEMINI_API_KEY found. StylistAgent will use a rule-based fallback.")


def log_event(agent_name: str, event: str, data: Optional[dict] = None):
    payload = {"agent": agent_name, "event": event, "data": data or {}}
    print(f"[LOG] {json.dumps(payload)}")


# ============================================================
# Cell 3: Product catalog + tools
# ============================================================

@dataclass
class Product:
    id: str
    name: str
    brand: str
    price: float
    image_url: str
    colors: List[str]
    categories: List[str]

# Use placeholder images so they always load.
# You can replace these URLs with your own product photos later.
PRODUCTS = [
    # ---------- Tops ----------
    Product(
        id="1",
        name="Pastel Linen Shirt",
        brand="Breeze",
        price=40.0,
        image_url="https://via.placeholder.com/400x400.png?text=Pastel+Linen+Shirt",
        colors=["pastel-pink", "white"],
        categories=["top", "smart-casual", "pastel"],
    ),
    Product(
        id="2",
        name="Cream Knit Sweater",
        brand="CozyHome",
        price=55.0,
        image_url="https://via.placeholder.com/400x400.png?text=Cream+Knit+Sweater",
        colors=["cream"],
        categories=["top", "casual", "neutral"],
    ),
    Product(
        id="3",
        name="White Cotton Tee",
        brand="Everyday",
        price=20.0,
        image_url="https://via.placeholder.com/400x400.png?text=White+Cotton+Tee",
        colors=["white"],
        categories=["top", "casual", "basic"],
    ),
    Product(
        id="4",
        name="Soft Blue Blouse",
        brand="Skyline",
        price=45.0,
        image_url="https://via.placeholder.com/400x400.png?text=Soft+Blue+Blouse",
        colors=["pastel-blue"],
        categories=["top", "smart-casual", "pastel"],
    ),

    # ---------- Bottoms ----------
    Product(
        id="5",
        name="White Chino Pants",
        brand="UrbanEase",
        price=45.0,
        image_url="https://via.placeholder.com/400x400.png?text=White+Chinos",
        colors=["white"],
        categories=["bottom", "smart-casual", "neutral"],
    ),
    Product(
        id="6",
        name="Beige Wide-Leg Trousers",
        brand="MinimalCo",
        price=60.0,
        image_url="https://via.placeholder.com/400x400.png?text=Beige+Trousers",
        colors=["beige"],
        categories=["bottom", "formal", "minimal"],
    ),
    Product(
        id="7",
        name="Light Wash Mom Jeans",
        brand="DenimDays",
        price=50.0,
        image_url="https://via.placeholder.com/400x400.png?text=Mom+Jeans",
        colors=["light-blue"],
        categories=["bottom", "casual"],
    ),

    # ---------- Dresses ----------
    Product(
        id="8",
        name="Pastel Maxi Dress",
        brand="Sunset Bloom",
        price=70.0,
        image_url="https://via.placeholder.com/400x400.png?text=Pastel+Maxi+Dress",
        colors=["pastel-blue", "pastel-pink"],
        categories=["dress", "occasion", "wedding", "pastel"],
    ),
    Product(
        id="9",
        name="Satin Slip Dress",
        brand="GlowWear",
        price=75.0,
        image_url="https://via.placeholder.com/400x400.png?text=Satin+Slip+Dress",
        colors=["champagne"],
        categories=["dress", "evening", "date"],
    ),
    Product(
        id="10",
        name="Floral Day Dress",
        brand="Gardenia",
        price=55.0,
        image_url="https://via.placeholder.com/400x400.png?text=Floral+Day+Dress",
        colors=["white", "pastel-pink"],
        categories=["dress", "casual", "day"],
    ),

    # ---------- Shoes ----------
    Product(
        id="11",
        name="Tan Loafers",
        brand="StepRight",
        price=50.0,
        image_url="https://via.placeholder.com/400x400.png?text=Tan+Loafers",
        colors=["tan"],
        categories=["shoes", "smart-casual"],
    ),
    Product(
        id="12",
        name="Strappy Sandals",
        brand="SoleWave",
        price=35.0,
        image_url="https://via.placeholder.com/400x400.png?text=Strappy+Sandals",
        colors=["beige"],
        categories=["shoes", "occasion", "summer"],
    ),
    Product(
        id="13",
        name="White Sneakers",
        brand="StreetStep",
        price=45.0,
        image_url="https://via.placeholder.com/400x400.png?text=White+Sneakers",
        colors=["white"],
        categories=["shoes", "casual"],
    ),

    # ---------- Accessories ----------
    Product(
        id="14",
        name="Statement Pearl Earrings",
        brand="Aurora",
        price=20.0,
        image_url="https://via.placeholder.com/400x400.png?text=Pearl+Earrings",
        colors=["white"],
        categories=["accessory", "occasion"],
    ),
    Product(
        id="15",
        name="Cream Tote Bag",
        brand="DailyCarry",
        price=40.0,
        image_url="https://via.placeholder.com/400x400.png?text=Cream+Tote+Bag",
        colors=["cream"],
        categories=["accessory", "casual", "minimal"],
    ),
    Product(
        id="16",
        name="Gold Layered Necklace",
        brand="GlowWear",
        price=22.0,
        image_url="https://via.placeholder.com/400x400.png?text=Gold+Necklace",
        colors=["gold"],
        categories=["accessory"],
    ),
]

print(f"Loaded {len(PRODUCTS)} demo products.")


def product_by_id(pid: str) -> Optional[Product]:
    for p in PRODUCTS:
        if p.id == pid:
            return p
    return None


def product_search(
    max_price: Optional[float] = None,
    min_price: Optional[float] = None,
    colors: Optional[List[str]] = None,
    categories: Optional[List[str]] = None,
    limit: int = 10,
) -> List[Product]:
    results = []
    for p in PRODUCTS:
        if max_price is not None and p.price > max_price:
            continue
        if min_price is not None and p.price < min_price:
            continue
        if colors and not any(c in p.colors for c in colors):
            continue
        if categories and not any(cat in p.categories for cat in categories):
            continue
        results.append(p)
    log_event("product_search", "filtered", {"count": len(results)})
    return results[:limit]


def color_harmony_score(items: List[Product]) -> float:
    if not items:
        return 0.0
    score = 0
    for p in items:
        for c in p.colors:
            if "pastel" in c or c in ["white", "beige", "tan", "cream"]:
                score += 2
            else:
                score += 1
    return min(1.0, score / (len(items) * 3))


# ============================================================
# Cell 4: Sessions & memory
# ============================================================

@dataclass
class UserPreferences:
    budget: Optional[float] = None
    styles: List[str] = field(default_factory=list)
    disliked_colors: List[str] = field(default_factory=list)
    preferred_fit: Optional[str] = None

@dataclass
class WardrobeItem:
    id: str
    name: str
    colors: List[str]
    category: str
    notes: str = ""

@dataclass
class SessionState:
    user_id: str
    preferences: UserPreferences = field(default_factory=UserPreferences)
    wardrobe: List[WardrobeItem] = field(default_factory=list)
    last_outfits: List[dict] = field(default_factory=list)
    saved_outfits: List[dict] = field(default_factory=list)  # saved looks

class SessionService:
    def __init__(self):
        self.sessions: Dict[str, SessionState] = {}

    def get_session(self, user_id: str) -> SessionState:
        if user_id not in self.sessions:
            self.sessions[user_id] = SessionState(user_id=user_id)
            log_event("SessionService", "create_session", {"user_id": user_id})
        return self.sessions[user_id]

    def update_preferences(self, user_id: str, **kwargs):
        s = self.get_session(user_id)
        for k, v in kwargs.items():
            setattr(s.preferences, k, v)
        log_event("SessionService", "update_preferences", {"user_id": user_id, "updates": kwargs})

session_service = SessionService()
DEMO_USER_ID = "demo_user"


# ============================================================
# Cell 5: Agents – Stylist, Budget, Closet, Trend
# ============================================================

class StylistAgent:
    """Uses Gemini if available; otherwise falls back to simple rules."""
    def __init__(self):
        self.model = genai.GenerativeModel("gemini-1.5-pro") if GEMINI_API_KEY else None

    def create_outfit(self, user_message: str, session: SessionState, candidate_products: List[Product]) -> dict:
        log_event("StylistAgent", "create_outfit_start", {"msg": user_message})
        if self.model is None:
            return self._fallback_outfit(user_message, session, candidate_products)

        products_json = [
            {
                "id": p.id,
                "name": p.name,
                "brand": p.brand,
                "price": p.price,
                "colors": p.colors,
                "categories": p.categories,
            }
            for p in candidate_products
        ]

        prompt = f"""
You are a professional fashion stylist AI.

User message:
\"\"\"{user_message}\"\"\"

User preferences:
- Budget: {session.preferences.budget}
- Styles: {", ".join(session.preferences.styles) if session.preferences.styles else "unknown"}
- Disliked colors: {", ".join(session.preferences.disliked_colors) if session.preferences.disliked_colors else "none"}

Catalog products (JSON):
{json.dumps(products_json[:25], indent=2)}

Select 3–5 items that form a cohesive outfit for the user's request.
Return ONLY valid JSON in this structure:

{{
  "name": "short outfit title",
  "description": "1–3 sentence description of the outfit and why it works",
  "item_ids": ["1", "2", "3"],
  "style_tags": ["tag1", "tag2"],
  "estimated_price": 123.45
}}
"""

        try:
            resp = self.model.generate_content(prompt)
            text = resp.text
            data = None
            try:
                data = json.loads(text)
            except Exception:
                start = text.find("{")
                end = text.rfind("}")
                if start != -1 and end != -1:
                    data = json.loads(text[start:end+1])
            if not data:
                log_event("StylistAgent", "parse_error_fallback", {})
                return self._fallback_outfit(user_message, session, candidate_products)
            log_event("StylistAgent", "create_outfit_success", {"item_ids": data.get("item_ids", [])})
            return data
        except Exception as e:
            log_event("StylistAgent", "error_fallback", {"error": str(e)})
            return self._fallback_outfit(user_message, session, candidate_products)

    def _fallback_outfit(self, user_message: str, session: SessionState, candidate_products: List[Product]) -> dict:
        items = []
        lower = user_message.lower()
        if "wedding" in lower:
            dresses = [p for p in candidate_products if "dress" in p.categories]
            shoes = [p for p in candidate_products if "shoes" in p.categories]
            accessories = [p for p in candidate_products if "accessory" in p.categories]
            if dresses:
                items.append(random.choice(dresses).id)
            if shoes:
                items.append(random.choice(shoes).id)
            if accessories:
                items.append(random.choice(accessories).id)
        else:
            # simple fallback: 1 top, 1 bottom, 1 shoes if possible
            tops = [p for p in candidate_products if "top" in p.categories]
            bottoms = [p for p in candidate_products if "bottom" in p.categories]
            shoes = [p for p in candidate_products if "shoes" in p.categories]
            if tops:
                items.append(random.choice(tops).id)
            if bottoms:
                items.append(random.choice(bottoms).id)
            if shoes:
                items.append(random.choice(shoes).id)
        total_price = sum(product_by_id(pid).price for pid in items)
        return {
            "name": "Simple Styled Look",
            "description": "A simple fallback outfit using matching items from the catalog.",
            "item_ids": items,
            "style_tags": ["fallback"],
            "estimated_price": total_price,
        }

class BudgetAgent:
    def optimize_outfit(self, outfit: dict, session: SessionState) -> dict:
        budget = session.preferences.budget
        if budget is None:
            return outfit
        ids = outfit["item_ids"]
        items = [product_by_id(pid) for pid in ids if product_by_id(pid)]
        total = sum(p.price for p in items)
        if total <= budget:
            outfit["budget_status"] = f"✅ Within budget (${total:.2f} of ${budget:.2f})"
            return outfit

        items_sorted = sorted(items, key=lambda p: p.price, reverse=True)
        for expensive in items_sorted:
            candidates = product_search(
                max_price=expensive.price - 10,
                categories=expensive.categories,
                limit=3,
            )
            if not candidates:
                continue
            cheaper = candidates[0]
            outfit["item_ids"].remove(expensive.id)
            outfit["item_ids"].append(cheaper.id)
            break

        new_items = [product_by_id(pid) for pid in outfit["item_ids"] if product_by_id(pid)]
        total_new = sum(p.price for p in new_items)
        status = "✅" if total_new <= budget else "⚠️"
        outfit["estimated_price"] = total_new
        outfit["budget_status"] = f"{status} Adjusted price: ${total_new:.2f} (budget ${budget:.2f})"
        return outfit

class ClosetAgent:
    def add_item(self, session: SessionState, name: str, colors: List[str], category: str, notes: str = ""):
        wid = f"w{len(session.wardrobe)+1}"
        item = WardrobeItem(id=wid, name=name, colors=colors, category=category, notes=notes)
        session.wardrobe.append(item)
        log_event("ClosetAgent", "add_item", {"id": wid})
        return item

    def list_items(self, session: SessionState) -> List[WardrobeItem]:
        return session.wardrobe

class TrendAgent:
    def __init__(self):
        self.trends = []
        self.last_refreshed = None

    def refresh_trends(self):
        log_event("TrendAgent", "refresh_start", {})
        time.sleep(1.0)
        now = datetime.utcnow().isoformat() + "Z"
        self.last_refreshed = now
        self.trends = [
            {
                "name": "Soft Pastel Wedding Guest",
                "vibe": "Light, romantic, flowy silhouettes in pastel tones.",
                "tags": ["pastel", "wedding", "romantic"],
            },
            {
                "name": "Minimal Resort Linen",
                "vibe": "Crisp whites and linens, relaxed tailoring, beach-perfect.",
                "tags": ["minimal", "linen", "resort"],
            },
            {
                "name": "Statement Accessories",
                "vibe": "Clean base outfits with bold earrings and bags.",
                "tags": ["accessories", "statement", "elevated-basics"],
            },
        ]
        log_event("TrendAgent", "refresh_done", {"count": len(self.trends)})
        return self.trends, now

    def get_trends(self):
        if not self.trends:
            trends, _ = self.refresh_trends()
            return trends
        return self.trends

trend_agent = TrendAgent()


# ============================================================
# Cell 6: Evaluation & visual helpers (cards / grids)
# ============================================================

def evaluate_outfit(outfit: dict, session: SessionState) -> dict:
    budget = session.preferences.budget or 200.0
    est = outfit.get("estimated_price", 0.0)
    if est <= budget:
        budget_fit = 1.0
    else:
        over_ratio = (est - budget) / max(budget, 1.0)
        budget_fit = max(0.0, 1.0 - over_ratio)
    items = [product_by_id(pid) for pid in outfit["item_ids"] if product_by_id(pid)]
    color_score = color_harmony_score(items)
    count = len(outfit["item_ids"])
    item_score = 1.0 if 3 <= count <= 5 else max(0.0, 1.0 - abs(count - 4)*0.25)
    overall = 0.4*budget_fit + 0.4*color_score + 0.2*item_score
    return {
        "budget_fit": round(budget_fit, 2),
        "color_harmony": round(color_score, 2),
        "item_count_score": round(item_score, 2),
        "overall": round(overall, 2),
    }

COLOR_MAP = {
    "pastel-pink": "#ffc1cc",
    "pastel-blue": "#c4e1ff",
    "white": "#ffffff",
    "cream": "#f5f5e8",
    "beige": "#f5f5dc",
    "tan": "#d2b48c",
    "light-blue": "#cde4ff",
    "gold": "#f3c969",
    "default": "#cccccc",
}

def color_to_hex(c: str) -> str:
    return COLOR_MAP.get(c.lower(), COLOR_MAP["default"])

def build_palette_html(items: list) -> str:
    colors = []
    for p in items:
        for c in p.get("colors", []):
            if c not in colors:
                colors.append(c)
    if not colors:
        return ""
    swatches = []
    for c in colors:
        swatches.append(
            f"<div style='display:inline-block;width:22px;height:22px;border-radius:999px;"
            f"margin-right:6px;background:{color_to_hex(c)};border:1px solid #e5e7eb;' title='{c}'></div>"
        )
    return "<div><strong>Palette:</strong><br>" + "".join(swatches) + "</div>"

def highlight_trends_for_outfit(outfit: dict) -> str:
    trends = trend_agent.get_trends()
    tags = set(outfit.get("style_tags", []))
    if not tags:
        return ""
    matches = []
    for t in trends:
        if tags.intersection(set(t["tags"])):
            matches.append(t["name"])
    if not matches:
        return ""
    lines = ["🔥 **Trend highlights:**"] + [f"- {m}" for m in matches]
    return "\n".join(lines)

def build_outfit_card_html(outfit: dict, items: list) -> str:
    name = outfit.get("name", "Styled Look")
    desc = outfit.get("description", "")
    scores = outfit.get("scores", {})
    est_price = outfit.get("estimated_price", 0.0)
    budget_status = outfit.get("budget_status", "")
    overall = scores.get("overall", None)
    color_score = scores.get("color_harmony", "?")
    score_str = f"<span style='font-weight:600;'>Overall:</span> {overall}" if overall is not None else ""
    return f"""
<div style="
    background: #ffffff;
    border-radius: 18px;
    padding: 12px 16px;
    border: 1px solid #f3e8ff;
    box-shadow: 0 10px 25px rgba(15, 23, 42, 0.06);
    color: #0f172a;
    font-family: system-ui, -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif;
">
  <div style="font-size: 16px; font-weight: 650; margin-bottom: 4px; color:#4b164c;">
    👗 {name}
  </div>
  <div style="font-size: 12px; opacity: 0.9; margin-bottom: 8px;">
    {desc}
  </div>
  <div style="font-size: 12px; margin-bottom: 2px;">
    <span style="font-weight:600;">Total:</span> ${est_price:.2f}
  </div>
  <div style="font-size: 11px; margin-bottom: 2px; color:#16a34a;">
    {budget_status}
  </div>
  <div style="font-size: 11px; opacity: 0.9;">
    {score_str} &nbsp; <span style="font-weight:600;">Color harmony:</span> {color_score}
  </div>
</div>
"""

def build_outfit_board_image(items: list):
    if not items:
        return None
    try:
        imgs = []
        target_h = 220
        for p in items:
            url = p.get("image_url")
            if not url:
                continue
            resp = requests.get(url, timeout=5)
            img = Image.open(BytesIO(resp.content)).convert("RGB")
            w, h = img.size
            new_w = int(w * (target_h / h))
            imgs.append(img.resize((new_w, target_h)))
        if not imgs:
            return None
        total_w = sum(img.size[0] for img in imgs)
        board = Image.new("RGB", (total_w, target_h), (250, 250, 255))
        x = 0
        for img in imgs:
            board.paste(img, (x, 0))
            x += img.size[0]
        return board
    except Exception as e:
        log_event("UI", "board_image_error", {"error": str(e)})
        return None

def build_products_html(items: list) -> str:
    if not items:
        return "<div style='font-size:13px; opacity:0.7;'>No products to show.</div>"
    cards = []
    for p in items:
        img_url = p.get("image_url", "")
        name = p.get("name", "Item")
        brand = p.get("brand", "")
        price = p.get("price", 0)
        colors = ", ".join(p.get("colors", []))
        cards.append(f"""
        <div style="
            min-width: 180px;
            max-width: 210px;
            background:#ffffff;
            border-radius:14px;
            border:1px solid #f3e8ff;
            box-shadow:0 6px 16px rgba(15,23,42,0.06);
            padding:10px;
            display:flex;
            flex-direction:column;
            gap:6px;
        ">
          <div style="width:100%;border-radius:10px;overflow:hidden;background:#f9fafb;">
            <img src="{img_url}" alt="{name}" style="width:100%;height:180px;object-fit:cover;display:block;" />
          </div>
          <div style="font-size:12px;font-weight:600;line-height:1.2;">
            {name}
          </div>
          <div style="font-size:11px;opacity:0.7;">
            {brand}
          </div>
          <div style="font-size:12px;font-weight:600;color:#4b164c;">
            ${price:.2f}
          </div>
          <div style="font-size:11px;opacity:0.7;">
            {colors}
          </div>
        </div>
        """)
    return f"""
<div style="overflow-x:auto;padding:4px 0 8px 0;">
  <div style="display:flex;flex-wrap:nowrap;gap:12px;">
    {''.join(cards)}
  </div>
</div>
"""

def build_closet_html(items: list) -> str:
    if not items:
        return "<div style='font-size:13px; opacity:0.7;'>Your closet is empty. Add a few favorites!</div>"
    cards = []
    for w in items:
        colors = ", ".join(w.colors)
        cards.append(f"""
        <div style="
            min-width: 180px;
            max-width: 220px;
            background:#ffffff;
            border-radius:14px;
            border:1px solid #f3e8ff;
            box-shadow:0 6px 16px rgba(15,23,42,0.06);
            padding:10px;
            display:flex;
            flex-direction:column;
            gap:4px;
        ">
          <div style="font-size:12px;font-weight:600;">
            {w.name}
          </div>
          <div style="font-size:11px;opacity:0.7;">
            Category: {w.category}
          </div>
          <div style="font-size:11px;opacity:0.7;">
            Colors: {colors}
          </div>
          <div style="font-size:11px;opacity:0.7;">
            {w.notes}
          </div>
        </div>
        """)
    return f"""
<div style="overflow-x:auto;padding:4px 0 8px 0;">
  <div style="display:flex;flex-wrap:nowrap;gap:12px;">
    {''.join(cards)}
  </div>
</div>
"""

def build_trends_html(trends: list) -> str:
    if not trends:
        return "<div style='font-size:13px; opacity:0.7;'>No trends yet.</div>"
    cards = []
    for t in trends:
        name = t.get("name", "Trend")
        vibe = t.get("vibe", "")
        tags = ", ".join(t.get("tags", []))
        cards.append(f"""
        <div style="
            min-width: 220px;
            max-width: 260px;
            background:#ffffff;
            border-radius:16px;
            border:1px solid #f3e8ff;
            box-shadow:0 6px 16px rgba(15,23,42,0.06);
            padding:12px;
            display:flex;
            flex-direction:column;
            gap:6px;
        ">
          <div style="font-size:13px;font-weight:650;color:#4b164c;">
            {name}
          </div>
          <div style="font-size:11px;opacity:0.85;">
            {vibe}
          </div>
          <div style="font-size:11px;opacity:0.8;">
            <span style="font-weight:600;">Tags:</span> {tags}
          </div>
        </div>
        """)
    return f"""
<div style="overflow-x:auto;padding:4px 0 8px 0;">
  <div style="display:flex;flex-wrap:nowrap;gap:12px;">
    {''.join(cards)}
  </div>
</div>
"""

def format_outfit_markdown(outfit: dict, items: list) -> str:
    lines = [f"**{outfit.get('name', 'Outfit')}**"]
    if outfit.get("description"):
        lines.append(outfit["description"])
    if outfit.get("estimated_price") is not None:
        lines.append(f"Total: `${outfit['estimated_price']:.2f}`")
    return "\n\n".join(lines)

def build_explainability(outfit: dict, items: list, session: SessionState) -> str:
    lines = ["### Why this works", ""]
    scores = outfit.get("scores", {})
    budget = session.preferences.budget
    est = outfit.get("estimated_price", 0.0)

    if budget is not None:
        if est <= budget:
            lines.append(f"- The outfit stays within your budget (${est:.2f} of ${budget:.2f}).")
        else:
            lines.append(f"- The look slightly exceeds your budget (${est:.2f} vs ${budget:.2f}) but keeps the overall vibe.")

    if "color_harmony" in scores:
        lines.append(f"- The color palette has a harmony score of **{scores['color_harmony']}**, so pieces blend nicely.")

    style_tags = outfit.get("style_tags", [])
    if style_tags:
        lines.append(f"- It follows your style tags: **{', '.join(style_tags)}**.")

    closet_colors = set()
    for w in session.wardrobe:
        closet_colors.update(w.colors)

    if closet_colors:
        overlap = set()
        for p in items:
            overlap |= closet_colors & set(p.get("colors", []))
        if overlap:
            lines.append(f"- Colors like **{', '.join(overlap)}** echo items in your closet, making this easy to re-wear.")

    if len(lines) <= 2:
        lines.append("- The pieces are selected to feel cohesive and wearable for your occasion.")

    return "\n".join(lines)


# ============================================================
# Closet-aware bias helper
# ============================================================

def apply_closet_bias(session: SessionState, candidates: List[Product]) -> List[Product]:
    """
    Boost products that share colors with the user's closet items.
    """
    closet_colors = set()
    for item in session.wardrobe:
        closet_colors.update(item.colors)
    if not closet_colors:
        return candidates
    return sorted(
        candidates,
        key=lambda p: len(closet_colors & set(p.colors)),
        reverse=True,
    )


# ============================================================
# Cell 7: Orchestrator
# ============================================================

class OrchestratorAgent:
    def __init__(self):
        self.stylist = StylistAgent()
        self.budget = BudgetAgent()
        self.closet = ClosetAgent()

    def classify_intent(self, message: str) -> str:
        msg = message.lower()
        if "show closet" in msg or "my wardrobe" in msg:
            return "SHOW_CLOSET"
        if "add to closet" in msg or "save this item" in msg:
            return "ADD_TO_CLOSET"
        return "OUTFIT_REQUEST"

    def handle_message(self, user_id: str, message: str) -> dict:
        session = session_service.get_session(user_id)
        intent = self.classify_intent(message)
        log_event("Orchestrator", "intent", {"intent": intent})

        if intent == "OUTFIT_REQUEST":
            return self._handle_outfit(session, message)
        elif intent == "SHOW_CLOSET":
            items = self.closet.list_items(session)
            return {"type": "closet", "text": f"You have {len(items)} items in your closet."}
        elif intent == "ADD_TO_CLOSET":
            item = self.closet.add_item(session, "User's white shirt", ["white"], "top", "Added from chat")
            return {"type": "closet_add", "text": f"Added {item.name} to your closet."}
        else:
            return {"type": "info", "text": "I can help you style outfits and manage your closet."}

    def _handle_outfit(self, session: SessionState, message: str) -> dict:
        words = message.lower().replace("$", "").split()
        nums = [w for w in words if w.replace(".", "", 1).isdigit()]
        if nums:
            session.preferences.budget = float(nums[0])
        candidates = product_search(max_price=session.preferences.budget, limit=20) if session.preferences.budget else PRODUCTS
        candidates = apply_closet_bias(session, candidates)

        outfit = self.stylist.create_outfit(message, session, candidates)
        outfit = self.budget.optimize_outfit(outfit, session)
        items = [product_by_id(pid).__dict__ for pid in outfit["item_ids"] if product_by_id(pid)]
        outfit["scores"] = evaluate_outfit(outfit, session)
        session.last_outfits.append(outfit)
        return {"type": "outfit", "outfit": outfit, "items": items}

orchestrator = OrchestratorAgent()


# ============================================================
# Cell 8: Trend, closet, saved outfits, catalog helpers + virtual try-on
# ============================================================

def load_trends_ui():
    trends = trend_agent.get_trends()
    ts = trend_agent.last_refreshed
    info = f"Last refreshed at: {ts}" if ts else "Trends not refreshed yet."
    html = build_trends_html(trends)
    return info, html

def refresh_trends_ui():
    trends, ts = trend_agent.refresh_trends()
    info = f"Last refreshed at: {ts}"
    html = build_trends_html(trends)
    return info, html

def show_closet_html():
    session = session_service.get_session(DEMO_USER_ID)
    items = session.wardrobe
    return build_closet_html(items)

def add_closet_item_ui(name, colors_text, category, notes):
    session = session_service.get_session(DEMO_USER_ID)
    if not name:
        return "Please provide a name.", show_closet_html()
    colors = [c.strip() for c in (colors_text or "").split(",") if c.strip()]
    item = orchestrator.closet.add_item(session, name, colors, category, notes)
    return f"Added '{item.name}' to your closet.", show_closet_html()

# Saved outfits
def build_saved_outfits_html(saved_list: List[dict]) -> str:
    if not saved_list:
        return "<div style='font-size:13px; opacity:0.7;'>You haven't saved any looks yet.</div>"

    cards = []
    for idx, entry in enumerate(saved_list, start=1):
        outfit = entry.get("outfit", {})
        items = entry.get("items", [])
        name = outfit.get("name", f"Look #{idx}")
        total = outfit.get("estimated_price", 0.0)
        count = len(items)
        tags = ", ".join(outfit.get("style_tags", []))

        cards.append(f"""
        <div style="
            min-width: 220px;
            max-width: 260px;
            background:#ffffff;
            border-radius:16px;
            border:1px solid #f3e8ff;
            box-shadow:0 6px 16px rgba(15,23,42,0.06);
            padding:12px;
            display:flex;
            flex-direction:column;
            gap:6px;
        ">
          <div style="font-size:13px;font-weight:650;color:#4b164c;">
            {name}
          </div>
          <div style="font-size:11px;opacity:0.85;">
            {count} piece(s) · Total ${total:.2f}
          </div>
          <div style="font-size:11px;opacity:0.8;">
            <span style="font-weight:600;">Tags:</span> {tags or '—'}
          </div>
        </div>
        """)

    return f"""
<div style="overflow-x:auto;padding:4px 0 8px 0;">
  <div style="display:flex;flex-wrap:nowrap;gap:12px;">
    {''.join(cards)}
  </div>
</div>
"""

def save_current_outfit_ui():
    session = session_service.get_session(DEMO_USER_ID)
    if not session.last_outfits:
        return "Style an outfit first before saving."
    last = session.last_outfits[-1]
    items = [product_by_id(pid).__dict__ for pid in last["item_ids"] if product_by_id(pid)]
    session.saved_outfits.append({"outfit": last, "items": items})
    log_event("SavedOutfit", "save", {"name": last.get("name", "Look")})
    return f"Saved '{last.get('name', 'Your look')}' to your Saved Looks."

def load_saved_outfits_ui():
    session = session_service.get_session(DEMO_USER_ID)
    return build_saved_outfits_html(session.saved_outfits)

# Browse catalog per category
def load_catalog_all():
    tops = [p.__dict__ for p in PRODUCTS if "top" in p.categories]
    bottoms = [p.__dict__ for p in PRODUCTS if "bottom" in p.categories]
    dresses = [p.__dict__ for p in PRODUCTS if "dress" in p.categories]
    shoes = [p.__dict__ for p in PRODUCTS if "shoes" in p.categories]
    accessories = [p.__dict__ for p in PRODUCTS if "accessory" in p.categories]

    tops_html = build_products_html(tops)
    bottoms_html = build_products_html(bottoms)
    dresses_html = build_products_html(dresses)
    shoes_html = build_products_html(shoes)
    accessories_html = build_products_html(accessories)

    return tops_html, bottoms_html, dresses_html, shoes_html, accessories_html

# Virtual Try-On helpers
def get_last_outfit_board(user_id: str = DEMO_USER_ID):
    session = session_service.get_session(user_id)
    if not session.last_outfits:
        return None, "No outfit yet. Ask the stylist first."
    last_outfit = session.last_outfits[-1]
    items = [product_by_id(pid).__dict__ for pid in last_outfit["item_ids"] if product_by_id(pid)]
    board = build_outfit_board_image(items)
    if board is None:
        return None, "Could not build outfit board."
    return board, f"Using your last outfit: {last_outfit.get('name', 'Styled Look')}"

def virtual_try_on(user_photo):
    if user_photo is None:
        return None, None, "Upload a photo first."
    board, info = get_last_outfit_board()
    return user_photo, board, info


# ============================================================
# Cell 9: style_chat backend
# ============================================================

def style_chat(message, chat_history, budget_slider, style_tags, event_select, tone_select):
    if chat_history is None:
        chat_history = []

    session = session_service.get_session(DEMO_USER_ID)
    if style_tags:
        session.preferences.styles = list(style_tags)
    if budget_slider and budget_slider > 0:
        session.preferences.budget = float(budget_slider)

    if event_select and event_select != "Auto-detect":
        message = f"{message} (Event: {event_select})"
    if tone_select and tone_select != "Any":
        message = f"{message} (Color tone: {tone_select})"

    result = orchestrator.handle_message(DEMO_USER_ID, message)

    reply_md = ""
    outfit_card_html = ""
    palette_html = ""
    trend_md = ""
    products_html = ""
    board_img = None

    if result["type"] == "outfit":
        outfit = result["outfit"]
        items = result["items"]
        base_md = format_outfit_markdown(outfit, items)
        explain_md = build_explainability(outfit, items, session)
        reply_md = base_md + "\n\n" + explain_md

        outfit_card_html = build_outfit_card_html(outfit, items)
        palette_html = build_palette_html(items)
        trend_md = highlight_trends_for_outfit(outfit)
        products_html = build_products_html(items)
        board_img = build_outfit_board_image(items)
    elif result["type"] == "closet":
        reply_md = result["text"]
    else:
        reply_md = result.get("text", "I can help you build outfits!")

    chat_history = chat_history + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": reply_md},
    ]

    return (
        chat_history,     # chatbot
        reply_md,         # outfit_md
        outfit_card_html, # outfit_card
        palette_html,     # palette_bar
        trend_md,         # trend_md
        products_html,    # products_html_comp
        board_img,        # board_image
    )


# ============================================================
# Cell 10: Gradio UI – fashion site style
# ============================================================

theme = gr.themes.Soft(
    primary_hue="pink",
    secondary_hue="indigo",
    radius_size="lg",
).set(
    body_background_fill="#faf5ff",
    body_text_color="#111827",
    block_background_fill="#ffffff",
    block_border_width="1px",
    block_border_color="#f3e8ff",
)

with gr.Blocks(title="StyleSphere AI – Personal Fashion Assistant", theme=theme) as demo:
    gr.Markdown(
        """
# 🌸 StyleSphere AI

A mini **boutique-style** personal fashion assistant:
Curated outfits · Your own closet · Trend-aware looks
"""
    )

    # ---------- TAB 1: AI Stylist ----------
    with gr.Tab("AI Stylist"):
        gr.Markdown("### 💬 Talk to your stylist")

        with gr.Row():
            with gr.Column(scale=3):
                chatbot = gr.Chatbot(
                    label="AI Stylist Chat",
                    height=320,
                    type="messages",
                )
            with gr.Column(scale=2):
                user_input = gr.Textbox(
                    label="Tell me what you need",
                    placeholder="e.g. I need an outfit for a beach wedding under $150. I like pastel colors.",
                )
                event_select = gr.Dropdown(
                    ["Auto-detect", "Wedding", "Office", "Casual", "Date", "Travel"],
                    value="Auto-detect",
                    label="Event",
                )
                style_tags = gr.CheckboxGroup(
                    ["pastel", "minimal", "streetwear", "formal", "boho"],
                    label="Style vibe",
                )
                budget_slider = gr.Slider(
                    minimum=0,
                    maximum=300,
                    value=0,
                    step=10,
                    label="Max budget (0 = no limit)",
                )
                tone_select = gr.Dropdown(
                    ["Any", "Light/Pastel", "Neutral", "Dark/Bold"],
                    value="Any",
                    label="Color tone",
                )
                send_btn = gr.Button("Style me ✨", variant="primary")

        gr.Markdown("### ⭐ Featured outfit")

        with gr.Row():
            with gr.Column(scale=2):
                outfit_card = gr.HTML(label="Outfit Card")
                palette_bar = gr.HTML(label="Color Palette")
            with gr.Column(scale=2):
                outfit_md = gr.Markdown(label="Quick notes & why it works")
                trend_md = gr.Markdown(label="Trend highlights")

        gr.Markdown("### 🛍️ Shop this look")

        products_html_comp = gr.HTML(label="Products")

        board_image = gr.Image(
            label="Outfit collage (preview)",
            interactive=False,
            height=220,
        )

        save_btn = gr.Button("Save this outfit ❤️")
        save_msg = gr.Markdown(label="Save status")

        def on_send(message, chat_history, budget_value, styles_value, event_value, tone_value):
            return style_chat(message, chat_history, budget_value, styles_value, event_value, tone_value)

        send_btn.click(
            on_send,
            inputs=[user_input, chatbot, budget_slider, style_tags, event_select, tone_select],
            outputs=[
                chatbot,
                outfit_md,
                outfit_card,
                palette_bar,
                trend_md,
                products_html_comp,
                board_image,
            ],
        )
        user_input.submit(
            on_send,
            inputs=[user_input, chatbot, budget_slider, style_tags, event_select, tone_select],
            outputs=[
                chatbot,
                outfit_md,
                outfit_card,
                palette_bar,
                trend_md,
                products_html_comp,
                board_image,
            ],
        )

        save_btn.click(
            save_current_outfit_ui,
            inputs=None,
            outputs=save_msg,
        )

    # ---------- TAB 2: Browse Catalog ----------
    with gr.Tab("Browse Catalog"):
        gr.Markdown("### 🛒 Shop by category")

        gr.Markdown("**Tops**")
        browse_tops = gr.HTML(label="Tops")

        gr.Markdown("**Bottoms**")
        browse_bottoms = gr.HTML(label="Bottoms")

        gr.Markdown("**Dresses**")
        browse_dresses = gr.HTML(label="Dresses")

        gr.Markdown("**Shoes**")
        browse_shoes = gr.HTML(label="Shoes")

        gr.Markdown("**Accessories**")
        browse_accessories = gr.HTML(label="Accessories")

        demo.load(
            load_catalog_all,
            inputs=None,
            outputs=[browse_tops, browse_bottoms, browse_dresses, browse_shoes, browse_accessories],
        )

    # ---------- TAB 3: My Closet ----------
    with gr.Tab("My Closet"):
        gr.Markdown("### 👚 Your Closet")

        with gr.Row():
            with gr.Column():
                closet_name = gr.Textbox(label="Item name", placeholder="e.g. White linen shirt")
                closet_colors = gr.Textbox(label="Colors (comma-separated)", placeholder="white, beige")
                closet_category = gr.Dropdown(
                    ["top", "bottom", "dress", "shoes", "accessory", "outerwear"],
                    label="Category",
                    value="top",
                )
                closet_notes = gr.Textbox(label="Notes (optional)")
                add_btn = gr.Button("Add to Closet")
                add_msg = gr.Markdown(label="Status")
            with gr.Column():
                closet_grid = gr.HTML(label="Closet items")

        demo.load(show_closet_html, inputs=None, outputs=closet_grid)

        add_btn.click(
            add_closet_item_ui,
            inputs=[closet_name, closet_colors, closet_category, closet_notes],
            outputs=[add_msg, closet_grid],
        )

    # ---------- TAB 4: Saved Looks ----------
    with gr.Tab("Saved Looks"):
        gr.Markdown("### 💖 Saved Looks")

        saved_grid = gr.HTML(label="Saved outfits")
        refresh_saved_btn = gr.Button("Refresh saved looks 🔄")

        demo.load(load_saved_outfits_ui, inputs=None, outputs=saved_grid)
        refresh_saved_btn.click(
            load_saved_outfits_ui,
            inputs=None,
            outputs=saved_grid,
        )

    # ---------- TAB 5: Trends ----------
    with gr.Tab("Trends"):
        gr.Markdown("### 🔮 Fashion Trends")

        trends_info = gr.Markdown(label="Refresh Info")
        trends_grid = gr.HTML(label="Trend grid")
        refresh_btn = gr.Button("Refresh Trends ♻️")

        demo.load(load_trends_ui, inputs=None, outputs=[trends_info, trends_grid])
        refresh_btn.click(
            refresh_trends_ui,
            inputs=None,
            outputs=[trends_info, trends_grid],
        )

    # ---------- TAB 6: Virtual Try-On ----------
    with gr.Tab("Virtual Try-On"):
        gr.Markdown(
            "### 🪞 Virtual Try-On (Preview)\n"
            "_Upload your photo and see it next to your latest styled outfit._"
        )

        with gr.Row():
            with gr.Column():
                user_photo = gr.Image(label="Upload your photo", type="pil")
                try_btn = gr.Button("Try my last outfit ✨")
                vt_caption = gr.Markdown(label="Info")
            with gr.Column():
                vt_user_preview = gr.Image(label="Your photo", interactive=False)
                vt_outfit_board = gr.Image(label="Outfit board", interactive=False)

        try_btn.click(
            virtual_try_on,
            inputs=[user_photo],
            outputs=[vt_user_preview, vt_outfit_board, vt_caption],
        )

demo.launch()


⚠️ No GEMINI_API_KEY found. StylistAgent will use a rule-based fallback.
Loaded 16 demo products.
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e2a03af717b1b678ef.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# ------------------ Add: recommend_matches backend ------------------

def score_candidate_combo(top: Product, bottom: Product, shoe: Product, accessory: Product, session: SessionState):
    """
    Simple scoring: 0.5 color harmony, 0.3 category-match (occasion), 0.2 price fit.
    Color harmony: reuse color_harmony_score on the trio.
    Category match: +1 if categories are aligned (both 'casual' or both 'smart-casual' etc.)
    Price fit: how close combined price is to top.price * 2 (heuristic).
    Returns score float and explain string.
    """
    items = [top, bottom, shoe, accessory]
    color_score = color_harmony_score(items)
    # category similarity heuristic
    top_cats = set(top.categories)
    bottom_cats = set(bottom.categories)
    shoe_cats = set(shoe.categories)
    accessory_cats = set(accessory.categories)
    common = 0
    for other in (bottom, shoe, accessory):
        if top_cats & set(other.categories):
            common += 1
    category_score = common / 3.0  # 0..1

    # price fit: aim for total between 1.5x-2.5x top price (heuristic)
    total = sum(p.price for p in items)
    target = top.price * 2.0
    price_diff = abs(total - target) / max(target, 1.0)
    price_score = max(0.0, 1.0 - price_diff)  # 1 if perfect, down to 0

    score = 0.5 * color_score + 0.3 * category_score + 0.2 * price_score
    # short explain
    explain = f"{top.name} pairs with {bottom.name}, {shoe.name} and {accessory.name} — color harmony {color_score:.2f}, vibe match {category_score:.2f}."
    return score, explain, total

def recommend_matches(product_id: str, user_id: str = DEMO_USER_ID, limit: int = 3):
    session = session_service.get_session(user_id)
    top = product_by_id(product_id)
    if not top:
        return []

    # Candidate pools
    bottoms = [p for p in PRODUCTS if "bottom" in p.categories]
    shoes = [p for p in PRODUCTS if "shoes" in p.categories]
    accessories = [p for p in PRODUCTS if "accessory" in p.categories]

    # Apply closet bias to each pool (promotes items sharing closet colors)
    bottoms = apply_closet_bias(session, bottoms)
    shoes = apply_closet_bias(session, shoes)
    accessories = apply_closet_bias(session, accessories)

    combos = []
    # keep it constrained: consider top 6 from each pool to avoid explosion
    for b in bottoms[:6]:
        for s in shoes[:6]:
            for a in accessories[:6]:
                score, explain, total = score_candidate_combo(top, b, s, a, session)
                combos.append({
                    "score": score,
                    "explain": explain,
                    "estimated_price": total,
                    "items": [b.id, s.id, a.id],
                    "items_full": [b.__dict__, s.__dict__, a.__dict__],
                })

    combos_sorted = sorted(combos, key=lambda c: c["score"], reverse=True)[:limit]
    # decorate with top info and small title/one-liner
    results = []
    for idx, c in enumerate(combos_sorted, start=1):
        title = f"Match #{idx}"
        one_liner = f"{top.name} + {c['items_full'][0]['name']} + {c['items_full'][1]['name']} — {c['explain'].split('—')[-1].strip()}"
        results.append({
            "title": title,
            "one_liner": one_liner,
            "item_ids": [top.id] + c["items"],
            "estimated_price": c["estimated_price"],
            "score": c["score"],
            "images": [top.image_url] + [it["image_url"] for it in c["items_full"]],
            "items_full": c["items_full"],
        })

    log_event("recommend_matches", "generated", {"top": top.id, "user": user_id, "count": len(results)})
    return results

# ------------------ Add: UI helper to render match cards as HTML ------------------

def build_match_cards_html(matches: list) -> str:
    if not matches:
        return "<div style='font-size:13px; opacity:0.7;'>No matches found.</div>"
    cards = []
    for m in matches:
        imgs_html = "".join([f"<img src='{u}' style='width:80px;height:80px;object-fit:cover;border-radius:8px;margin-right:6px;' />" for u in m["images"]])
        cards.append(f"""
        <div style="min-width:240px;max-width:260px;background:#fff;border-radius:14px;border:1px solid #f3e8ff;padding:10px;display:flex;flex-direction:column;gap:8px;">
          <div style="font-weight:650;color:#4b164c;">{m['title']}</div>
          <div style="font-size:12px;opacity:0.9;">{m['one_liner']}</div>
          <div style="display:flex;align-items:center;margin-top:6px;">{imgs_html}</div>
          <div style="display:flex;justify-content:space-between;align-items:center;margin-top:8px;">
            <div style="font-weight:600;color:#4b164c;">${m['estimated_price']:.2f}</div>
            <div>
              <button data-itemids='{','.join(m['item_ids'])}' style="background:#4b164c;color:#fff;border-radius:8px;padding:6px 8px;border:none;cursor:pointer;">Add all</button>
            </div>
          </div>
        </div>
        """)
    return f"<div style='overflow-x:auto;padding:4px 0 8px 0;'><div style='display:flex;gap:12px;'>{''.join(cards)}</div></div>"

# ------------------ Add: Gradio UI wiring for Browse Catalog ------------------
# Modify your Browse Catalog tab UI to include:
# - a dropdown of tops (top_select)
# - a button Find matches (find_matches_btn)
# - an HTML output matches_html

# Example placement inside the Browse Catalog tab (replace / augment existing content):
# Add these components where you want the matching UI to appear.
# Then wire the .click below.

def load_top_options():
    tops = [p for p in PRODUCTS if "top" in p.categories]
    return [(p.id, f"{p.name} — ${p.price:.2f}") for p in tops]

def recommend_matches_ui(top_id):
    if not top_id:
        return "Please select a top first.", "<div style='font-size:13px; opacity:0.7;'>No matches yet.</div>"
    matches = recommend_matches(top_id, DEMO_USER_ID, limit=3)
    html = build_match_cards_html(matches)
    return f"Showing matches for top id {top_id}", html


In [ ]:
# Enhanced StyleSphere AI - single-file Gradio app
# Integrates recommend_matches flow and UI improvements as requested.
# Note: requires environment variable GEMINI_API_KEY if you want Gemini; otherwise uses fallbacks.

# pip install google-generativeai gradio pillow requests

import os
import json
import random
import time
from datetime import datetime
from dataclasses import dataclass, field
from typing import List, Dict, Optional

import requests
from io import BytesIO
from PIL import Image

import google.generativeai as genai
import gradio as gr

# ---------------- Gemini config ----------------
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", None)
if GEMINI_API_KEY:
    genai.configure(api_key=GEMINI_API_KEY)
    print("✅ Gemini configured from environment variable.")
else:
    print("⚠️ No GEMINI_API_KEY found. StylistAgent will use a rule-based fallback.")

# ---------------- Utils ----------------
def log_event(agent_name: str, event: str, data: Optional[dict] = None):
    payload = {"agent": agent_name, "event": event, "data": data or {}}
    print(f"[LOG] {json.dumps(payload)}")

# ---------------- Data models & products ----------------
@dataclass
class Product:
    id: str
    name: str
    brand: str
    price: float
    image_url: str
    colors: List[str]
    categories: List[str]

PRODUCTS = [
    Product(id="1", name="Pastel Linen Shirt", brand="Breeze", price=40.0,
            image_url="https://via.placeholder.com/400x400.png?text=Pastel+Linen+Shirt", colors=["pastel-pink","white"], categories=["top","smart-casual","pastel"]),
    Product(id="2", name="Cream Knit Sweater", brand="CozyHome", price=55.0,
            image_url="https://via.placeholder.com/400x400.png?text=Cream+Knit+Sweater", colors=["cream"], categories=["top","casual","neutral"]),
    Product(id="3", name="White Cotton Tee", brand="Everyday", price=20.0,
            image_url="https://via.placeholder.com/400x400.png?text=White+Cotton+Tee", colors=["white"], categories=["top","casual","basic"]),
    Product(id="4", name="Soft Blue Blouse", brand="Skyline", price=45.0,
            image_url="https://via.placeholder.com/400x400.png?text=Soft+Blue+Blouse", colors=["pastel-blue"], categories=["top","smart-casual","pastel"]),
    Product(id="5", name="White Chino Pants", brand="UrbanEase", price=45.0,
            image_url="https://via.placeholder.com/400x400.png?text=White+Chinos", colors=["white"], categories=["bottom","smart-casual","neutral"]),
    Product(id="6", name="Beige Wide-Leg Trousers", brand="MinimalCo", price=60.0,
            image_url="https://via.placeholder.com/400x400.png?text=Beige+Trousers", colors=["beige"], categories=["bottom","formal","minimal"]),
    Product(id="7", name="Light Wash Mom Jeans", brand="DenimDays", price=50.0,
            image_url="https://via.placeholder.com/400x400.png?text=Mom+Jeans", colors=["light-blue"], categories=["bottom","casual"]),
    Product(id="8", name="Pastel Maxi Dress", brand="Sunset Bloom", price=70.0,
            image_url="https://via.placeholder.com/400x400.png?text=Pastel+Maxi+Dress", colors=["pastel-blue","pastel-pink"], categories=["dress","occasion","wedding","pastel"]),
    Product(id="9", name="Satin Slip Dress", brand="GlowWear", price=75.0,
            image_url="https://via.placeholder.com/400x400.png?text=Satin+Slip+Dress", colors=["champagne"], categories=["dress","evening","date"]),
    Product(id="10", name="Floral Day Dress", brand="Gardenia", price=55.0,
            image_url="https://via.placeholder.com/400x400.png?text=Floral+Day+Dress", colors=["white","pastel-pink"], categories=["dress","casual","day"]),
    Product(id="11", name="Tan Loafers", brand="StepRight", price=50.0,
            image_url="https://via.placeholder.com/400x400.png?text=Tan+Loafers", colors=["tan"], categories=["shoes","smart-casual"]),
    Product(id="12", name="Strappy Sandals", brand="SoleWave", price=35.0,
            image_url="https://via.placeholder.com/400x400.png?text=Strappy+Sandals", colors=["beige"], categories=["shoes","occasion","summer"]),
    Product(id="13", name="White Sneakers", brand="StreetStep", price=45.0,
            image_url="https://via.placeholder.com/400x400.png?text=White+Sneakers", colors=["white"], categories=["shoes","casual"]),
    Product(id="14", name="Statement Pearl Earrings", brand="Aurora", price=20.0,
            image_url="https://via.placeholder.com/400x400.png?text=Pearl+Earrings", colors=["white"], categories=["accessory","occasion"]),
    Product(id="15", name="Cream Tote Bag", brand="DailyCarry", price=40.0,
            image_url="https://via.placeholder.com/400x400.png?text=Cream+Tote+Bag", colors=["cream"], categories=["accessory","casual","minimal"]),
    Product(id="16", name="Gold Layered Necklace", brand="GlowWear", price=22.0,
            image_url="https://via.placeholder.com/400x400.png?text=Gold+Necklace", colors=["gold"], categories=["accessory"]),
]

print(f"Loaded {len(PRODUCTS)} demo products.")

# ---------------- Helpers ----------------
def product_by_id(pid: str) -> Optional[Product]:
    for p in PRODUCTS:
        if p.id == pid:
            return p
    return None


def product_search(max_price: Optional[float] = None, min_price: Optional[float] = None,
                   colors: Optional[List[str]] = None, categories: Optional[List[str]] = None, limit: int = 10) -> List[Product]:
    results = []
    for p in PRODUCTS:
        if max_price is not None and p.price > max_price:
            continue
        if min_price is not None and p.price < min_price:
            continue
        if colors and not any(c in p.colors for c in colors):
            continue
        if categories and not any(cat in p.categories for cat in categories):
            continue
        results.append(p)
    log_event("product_search", "filtered", {"count": len(results)})
    return results[:limit]


def color_harmony_score(items: List[Product]) -> float:
    if not items:
        return 0.0
    score = 0
    for p in items:
        for c in p.colors:
            if "pastel" in c or c in ["white", "beige", "tan", "cream"]:
                score += 2
            else:
                score += 1
    return min(1.0, score / (len(items) * 3))

# ---------------- Session & memory ----------------
@dataclass
class UserPreferences:
    budget: Optional[float] = None
    styles: List[str] = field(default_factory=list)
    disliked_colors: List[str] = field(default_factory=list)
    preferred_fit: Optional[str] = None

@dataclass
class WardrobeItem:
    id: str
    name: str
    colors: List[str]
    category: str
    notes: str = ""

@dataclass
class SessionState:
    user_id: str
    preferences: UserPreferences = field(default_factory=UserPreferences)
    wardrobe: List[WardrobeItem] = field(default_factory=list)
    last_outfits: List[dict] = field(default_factory=list)
    saved_outfits: List[dict] = field(default_factory=list)

class SessionService:
    def __init__(self):
        self.sessions: Dict[str, SessionState] = {}
    def get_session(self, user_id: str) -> SessionState:
        if user_id not in self.sessions:
            self.sessions[user_id] = SessionState(user_id=user_id)
            log_event("SessionService", "create_session", {"user_id": user_id})
        return self.sessions[user_id]
    def update_preferences(self, user_id: str, **kwargs):
        s = self.get_session(user_id)
        for k, v in kwargs.items():
            setattr(s.preferences, k, v)
        log_event("SessionService", "update_preferences", {"user_id": user_id, "updates": kwargs})

session_service = SessionService()
DEMO_USER_ID = "demo_user"

# ---------------- Agents (simplified stylist using fallback) ----------------
class StylistAgent:
    def __init__(self):
        self.model = genai.GenerativeModel("gemini-1.5-pro") if GEMINI_API_KEY else None
    def create_outfit(self, user_message: str, session: SessionState, candidate_products: List[Product]) -> dict:
        log_event("StylistAgent", "create_outfit_start", {"msg": user_message})
        # For this enhanced demo we rely on fallback to keep behavior deterministic
        return self._fallback_outfit(user_message, session, candidate_products)
    def _fallback_outfit(self, user_message: str, session: SessionState, candidate_products: List[Product]) -> dict:
        items = []
        lower = user_message.lower()
        if "wedding" in lower:
            dresses = [p for p in candidate_products if "dress" in p.categories]
            shoes = [p for p in candidate_products if "shoes" in p.categories]
            accessories = [p for p in candidate_products if "accessory" in p.categories]
            if dresses:
                items.append(random.choice(dresses).id)
            if shoes:
                items.append(random.choice(shoes).id)
            if accessories:
                items.append(random.choice(accessories).id)
        else:
            tops = [p for p in candidate_products if "top" in p.categories]
            bottoms = [p for p in candidate_products if "bottom" in p.categories]
            shoes = [p for p in candidate_products if "shoes" in p.categories]
            if tops:
                items.append(random.choice(tops).id)
            if bottoms:
                items.append(random.choice(bottoms).id)
            if shoes:
                items.append(random.choice(shoes).id)
        total_price = sum(product_by_id(pid).price for pid in items)
        return {"name": "Simple Styled Look", "description": "A simple fallback outfit using matching items from the catalog.", "item_ids": items, "style_tags": ["fallback"], "estimated_price": total_price}

class BudgetAgent:
    def optimize_outfit(self, outfit: dict, session: SessionState) -> dict:
        budget = session.preferences.budget
        if budget is None:
            return outfit
        ids = outfit["item_ids"]
        items = [product_by_id(pid) for pid in ids if product_by_id(pid)]
        total = sum(p.price for p in items)
        if total <= budget:
            outfit["budget_status"] = f"✅ Within budget (${total:.2f} of ${budget:.2f})"
            return outfit
        items_sorted = sorted(items, key=lambda p: p.price, reverse=True)
        for expensive in items_sorted:
            candidates = product_search(max_price=expensive.price - 10, categories=expensive.categories, limit=3)
            if not candidates:
                continue
            cheaper = candidates[0]
            outfit["item_ids"].remove(expensive.id)
            outfit["item_ids"].append(cheaper.id)
            break
        new_items = [product_by_id(pid) for pid in outfit["item_ids"] if product_by_id(pid)]
        total_new = sum(p.price for p in new_items)
        status = "✅" if total_new <= budget else "⚠️"
        outfit["estimated_price"] = total_new
        outfit["budget_status"] = f"{status} Adjusted price: ${total_new:.2f} (budget ${budget:.2f})"
        return outfit

class ClosetAgent:
    def add_item(self, session: SessionState, name: str, colors: List[str], category: str, notes: str = ""):
        wid = f"w{len(session.wardrobe)+1}"
        item = WardrobeItem(id=wid, name=name, colors=colors, category=category, notes=notes)
        session.wardrobe.append(item)
        log_event("ClosetAgent", "add_item", {"id": wid})
        return item
    def list_items(self, session: SessionState) -> List[WardrobeItem]:
        return session.wardrobe

class TrendAgent:
    def __init__(self):
        self.trends = []
        self.last_refreshed = None
    def refresh_trends(self):
        log_event("TrendAgent", "refresh_start", {})
        time.sleep(0.3)
        now = datetime.utcnow().isoformat() + "Z"
        self.last_refreshed = now
        self.trends = [
            {"name": "Soft Pastel Wedding Guest", "vibe": "Light, romantic, flowy silhouettes in pastel tones.", "tags": ["pastel","wedding","romantic"]},
            {"name": "Minimal Resort Linen", "vibe": "Crisp whites and linens, relaxed tailoring, beach-perfect.", "tags": ["minimal","linen","resort"]},
            {"name": "Statement Accessories", "vibe": "Clean base outfits with bold earrings and bags.", "tags": ["accessories","statement","elevated-basics"]},
        ]
        log_event("TrendAgent", "refresh_done", {"count": len(self.trends)})
        return self.trends, now
    def get_trends(self):
        if not self.trends:
            trends, _ = self.refresh_trends()
            return trends
        return self.trends

trend_agent = TrendAgent()

# ---------------- Evaluation & UI helpers ----------------
COLOR_MAP = {"pastel-pink":"#ffc1cc","pastel-blue":"#c4e1ff","white":"#ffffff","cream":"#f5f5e8","beige":"#f5f5dc","tan":"#d2b48c","light-blue":"#cde4ff","gold":"#f3c969","default":"#cccccc"}

def color_to_hex(c: str) -> str:
    return COLOR_MAP.get(c.lower(), COLOR_MAP["default"])

# reuse build functions from original; smaller for brevity

def build_products_html(items: list) -> str:
    if not items:
        return "<div style='font-size:13px; opacity:0.7;'>No products to show.</div>"
    cards = []
    for p in items:
        img_url = p.get("image_url", "")
        name = p.get("name", "Item")
        brand = p.get("brand", "")
        price = p.get("price", 0)
        colors = ", ".join(p.get("colors", []))
        cards.append(f"""
        <div style="min-width: 180px; max-width: 210px; background:#ffffff; border-radius:14px; border:1px solid #f3e8ff; box-shadow:0 6px 16px rgba(15,23,42,0.06); padding:10px; display:flex; flex-direction:column; gap:6px;">
          <div style="width:100%;border-radius:10px;overflow:hidden;background:#f9fafb;"><img src=\"{img_url}\" alt=\"{name}\" style=\"width:100%;height:180px;object-fit:cover;display:block;\" /></div>
          <div style="font-size:12px;font-weight:600;line-height:1.2;">{name}</div>
          <div style="font-size:11px;opacity:0.7;">{brand}</div>
          <div style="font-size:12px;font-weight:600;color:#4b164c;">${price:.2f}</div>
          <div style="font-size:11px;opacity:0.7;">{colors}</div>
        </div>
        """)
    return f"<div style=\"overflow-x:auto;padding:4px 0 8px 0;\"><div style=\"display:flex;flex-wrap:nowrap;gap:12px;\">{''.join(cards)}</div></div>"

# build outfit collage (board)

def build_outfit_board_image(items: list):
    if not items:
        return None
    try:
        imgs = []
        target_h = 220
        for p in items:
            url = p.get("image_url")
            if not url:
                continue
            resp = requests.get(url, timeout=5)
            img = Image.open(BytesIO(resp.content)).convert("RGB")
            w, h = img.size
            new_w = int(w * (target_h / h))
            imgs.append(img.resize((new_w, target_h)))
        if not imgs:
            return None
        total_w = sum(img.size[0] for img in imgs)
        board = Image.new("RGB", (total_w, target_h), (250, 250, 255))
        x = 0
        for img in imgs:
            board.paste(img, (x, 0))
            x += img.size[0]
        return board
    except Exception as e:
        log_event("UI", "board_image_error", {"error": str(e)})
        return None

# ---------------- Closet bias ----------------
def apply_closet_bias(session: SessionState, candidates: List[Product]) -> List[Product]:
    closet_colors = set()
    for item in session.wardrobe:
        closet_colors.update(item.colors)
    if not closet_colors:
        return candidates
    return sorted(candidates, key=lambda p: len(closet_colors & set(p.colors)), reverse=True)

# ---------------- Orchestrator simplified ----------------
class OrchestratorAgent:
    def __init__(self):
        self.stylist = StylistAgent()
        self.budget = BudgetAgent()
        self.closet = ClosetAgent()
    def handle_message(self, user_id: str, message: str) -> dict:
        session = session_service.get_session(user_id)
        candidates = product_search(max_price=session.preferences.budget, limit=20) if session.preferences.budget else PRODUCTS
        candidates = apply_closet_bias(session, candidates)
        outfit = self.stylist.create_outfit(message, session, candidates)
        outfit = self.budget.optimize_outfit(outfit, session)
        items = [product_by_id(pid).__dict__ for pid in outfit["item_ids"] if product_by_id(pid)]
        outfit["scores"] = {"overall": 0.8, "color_harmony": 0.9}
        session.last_outfits.append(outfit)
        return {"type": "outfit", "outfit": outfit, "items": items}

orchestrator = OrchestratorAgent()

# ---------------- Recommend matches implementation (new) ----------------

def score_candidate_combo(top: Product, bottom: Product, shoe: Product, accessory: Product, session: SessionState):
    items = [top, bottom, shoe, accessory]
    color_score = color_harmony_score(items)
    top_cats = set(top.categories)
    common = 0
    for other in (bottom, shoe, accessory):
        if top_cats & set(other.categories):
            common += 1
    category_score = common / 3.0
    total = sum(p.price for p in items)
    target = top.price * 2.0
    price_diff = abs(total - target) / max(target, 1.0)
    price_score = max(0.0, 1.0 - price_diff)
    score = 0.5 * color_score + 0.3 * category_score + 0.2 * price_score
    explain = f"{top.name} pairs with {bottom.name}, {shoe.name} and {accessory.name} — color harmony {color_score:.2f}, vibe match {category_score:.2f}."
    return score, explain, total


def recommend_matches(product_id: str, user_id: str = DEMO_USER_ID, limit: int = 3):
    session = session_service.get_session(user_id)
    top = product_by_id(product_id)
    if not top:
        return []
    bottoms = [p for p in PRODUCTS if "bottom" in p.categories]
    shoes = [p for p in PRODUCTS if "shoes" in p.categories]
    accessories = [p for p in PRODUCTS if "accessory" in p.categories]
    bottoms = apply_closet_bias(session, bottoms)
    shoes = apply_closet_bias(session, shoes)
    accessories = apply_closet_bias(session, accessories)
    combos = []
    for b in bottoms[:6]:
        for s in shoes[:6]:
            for a in accessories[:6]:
                score, explain, total = score_candidate_combo(top, b, s, a, session)
                combos.append({"score": score, "explain": explain, "estimated_price": total, "items": [b.id, s.id, a.id], "items_full": [b.__dict__, s.__dict__, a.__dict__]})
    combos_sorted = sorted(combos, key=lambda c: c["score"], reverse=True)[:limit]
    results = []
    for idx, c in enumerate(combos_sorted, start=1):
        title = f"Match #{idx}"
        one_liner = f"{top.name} + {c['items_full'][0]['name']} + {c['items_full'][1]['name']} — {c['explain'].split('—')[-1].strip()}"
        results.append({"title": title, "one_liner": one_liner, "item_ids": [top.id] + c["items"], "estimated_price": c["estimated_price"], "score": c["score"], "images": [top.image_url] + [it["image_url"] for it in c["items_full"]], "items_full": c["items_full"]})
    log_event("recommend_matches", "generated", {"top": top.id, "user": user_id, "count": len(results)})
    return results


def build_match_cards_html(matches: list) -> str:
    if not matches:
        return "<div style='font-size:13px; opacity:0.7;'>No matches found.</div>"
    cards = []
    for idx, m in enumerate(matches, start=1):
        imgs_html = "".join([f"<img src='{u}' style='width:80px;height:80px;object-fit:cover;border-radius:8px;margin-right:6px;' />" for u in m['images']])
        cards.append(f"""
        <div style="min-width:240px;max-width:260px;background:#fff;border-radius:14px;border:1px solid #f3e8ff;padding:10px;display:flex;flex-direction:column;gap:8px;">
          <div style="font-weight:650;color:#4b164c;">{m['title']}</div>
          <div style="font-size:12px;opacity:0.9;">{m['one_liner']}</div>
          <div style="display:flex;align-items:center;margin-top:6px;">{imgs_html}</div>
          <div style="display:flex;justify-content:space-between;align-items:center;margin-top:8px;">
            <div style="font-weight:600;color:#4b164c;">${m['estimated_price']:.2f}</div>
            <div style="font-size:12px;opacity:0.85;">Score: {m['score']:.2f}</div>
          </div>
        </div>
        """)
    return f"<div style='overflow-x:auto;padding:4px 0 8px 0;'><div style='display:flex;gap:12px;'>{''.join(cards)}</div></div>"

# ---------------- Saved outfits helpers ----------------

def save_match_as_outfit(user_id: str, match: dict):
    session = session_service.get_session(user_id)
    items = [product_by_id(pid).__dict__ for pid in match["item_ids"] if product_by_id(pid)]
    session.saved_outfits.append({"outfit": {"name": match.get("title", "Matched Look"), "item_ids": match["item_ids"], "estimated_price": match.get("estimated_price", 0.0), "style_tags": []}, "items": items})
    log_event("SavedMatch", "save", {"user": user_id, "title": match.get("title")})
    return f"Saved '{match.get('title', 'Look')}' to your Saved Looks."

# ---------------- Gradio UI ----------------

def load_catalog_all():
    tops = [p.__dict__ for p in PRODUCTS if "top" in p.categories]
    bottoms = [p.__dict__ for p in PRODUCTS if "bottom" in p.categories]
    dresses = [p.__dict__ for p in PRODUCTS if "dress" in p.categories]
    shoes = [p.__dict__ for p in PRODUCTS if "shoes" in p.categories]
    accessories = [p.__dict__ for p in PRODUCTS if "accessory" in p.categories]
    tops_html = build_products_html(tops)
    bottoms_html = build_products_html(bottoms)
    dresses_html = build_products_html(dresses)
    shoes_html = build_products_html(shoes)
    accessories_html = build_products_html(accessories)
    return tops_html, bottoms_html, dresses_html, shoes_html, accessories_html

# Functions for UI actions

def load_top_options():
    tops = [p for p in PRODUCTS if "top" in p.categories]
    return [(p.id, f"{p.name} — ${p.price:.2f}") for p in tops]


def recommend_matches_ui(top_id):
    if not top_id:
        return "Please select a top first.", "<div style='font-size:13px; opacity:0.7;'>No matches yet.</div>", []
    matches = recommend_matches(top_id, DEMO_USER_ID, limit=3)
    html = build_match_cards_html(matches)
    # also return choices for a match dropdown and the matches data as JSON
    choices = [m['title'] for m in matches]
    return f"Showing {len(matches)} match(es)", html, matches


def add_selected_match_to_saved(match_index: int, matches_state: list):
    if not matches_state:
        return "No matches to save. Generate matches first."
    if match_index is None or match_index < 0 or match_index >= len(matches_state):
        return "Please select a valid match."
    match = matches_state[match_index]
    return save_match_as_outfit(DEMO_USER_ID, match)

# ---------------- Gradio app layout ----------------

theme = gr.themes.Soft(primary_hue="pink", secondary_hue="indigo", radius_size="lg").set(
    body_background_fill="#faf5ff",
    body_text_color="#111827",
    block_background_fill="#ffffff",
    block_border_width="1px",
    block_border_color="#f3e8ff",
)

with gr.Blocks(title="StyleSphere AI – Enhanced Boutique Demo", theme=theme) as demo:
    gr.Markdown("""
# 🌸 StyleSphere AI — Enhanced Demo
Now includes: **Match this top → curated look suggestions**, save matched looks, and better catalog browsing.
""")

    with gr.Tab("AI Stylist"):
        gr.Markdown("### 💬 Talk to your stylist")
        with gr.Row():
            with gr.Column(scale=3):
                chatbot = gr.Chatbot(label="AI Stylist Chat", height=320, type="messages")
            with gr.Column(scale=2):
                user_input = gr.Textbox(label="Tell me what you need", placeholder="e.g. I need an outfit for a beach wedding under $150. I like pastel colors.")
                event_select = gr.Dropdown(["Auto-detect","Wedding","Office","Casual","Date","Travel"], value="Auto-detect", label="Event")
                style_tags = gr.CheckboxGroup(["pastel","minimal","streetwear","formal","boho"], label="Style vibe")
                budget_slider = gr.Slider(minimum=0, maximum=300, value=0, step=10, label="Max budget (0 = no limit)")
                tone_select = gr.Dropdown(["Any","Light/Pastel","Neutral","Dark/Bold"], value="Any", label="Color tone")
                send_btn = gr.Button("Style me ✨", variant="primary")
        gr.Markdown("### ⭐ Featured outfit")
        with gr.Row():
            with gr.Column(scale=2):
                outfit_card = gr.HTML(label="Outfit Card")
                palette_bar = gr.HTML(label="Color Palette")
            with gr.Column(scale=2):
                outfit_md = gr.Markdown(label="Quick notes & why it works")
                trend_md = gr.Markdown(label="Trend highlights")
        gr.Markdown("### 🛍️ Shop this look")
        products_html_comp = gr.HTML(label="Products")
        board_image = gr.Image(label="Outfit collage (preview)", interactive=False, height=220)

    with gr.Tab("Browse Catalog"):
        gr.Markdown("### 🛒 Shop by category & match a top")
        gr.Markdown("**Select a top to find curated matches (bottom + shoes + accessory)**")
        with gr.Row():
            with gr.Column(scale=2):
                browse_tops = gr.HTML(label="Tops")
                top_select = gr.Dropdown(choices=[], label="Select a top to find matches", value=None)
                find_matches_btn = gr.Button("Find matches for this top")
                matches_info = gr.Markdown(label="Matches info")
            with gr.Column(scale=2):
                matches_html = gr.HTML(label="Matches")
                match_choice = gr.Dropdown(choices=[], label="Choose a match to save", value=None)
                save_match_btn = gr.Button("Save selected match ❤️")
                save_match_msg = gr.Markdown(label="Save status")
        gr.Markdown("**Catalog**")
        browse_bottoms = gr.HTML(label="Bottoms")
        browse_dresses = gr.HTML(label="Dresses")
        browse_shoes = gr.HTML(label="Shoes")
        browse_accessories = gr.HTML(label="Accessories")

    with gr.Tab("My Closet"):
        gr.Markdown("### 👚 Your Closet")
        with gr.Row():
            with gr.Column():
                closet_name = gr.Textbox(label="Item name", placeholder="e.g. White linen shirt")
                closet_colors = gr.Textbox(label="Colors (comma-separated)", placeholder="white, beige")
                closet_category = gr.Dropdown(["top","bottom","dress","shoes","accessory","outerwear"], label="Category", value="top")
                closet_notes = gr.Textbox(label="Notes (optional)")
                add_btn = gr.Button("Add to Closet")
                add_msg = gr.Markdown(label="Status")
            with gr.Column():
                closet_grid = gr.HTML(label="Closet items")

    with gr.Tab("Saved Looks"):
        gr.Markdown("### 💖 Saved Looks")
        saved_grid = gr.HTML(label="Saved outfits")
        refresh_saved_btn = gr.Button("Refresh saved looks 🔄")

    with gr.Tab("Trends"):
        gr.Markdown("### 🔮 Fashion Trends")
        trends_info = gr.Markdown(label="Refresh Info")
        trends_grid = gr.HTML(label="Trend grid")
        refresh_btn = gr.Button("Refresh Trends ♻️")

    with gr.Tab("Virtual Try-On"):
        gr.Markdown("### 🪞 Virtual Try-On (Preview)")
        with gr.Row():
            with gr.Column():
                user_photo = gr.Image(label="Upload your photo", type="pil")
                try_btn = gr.Button("Try my last outfit ✨")
                vt_caption = gr.Markdown(label="Info")
            with gr.Column():
                vt_user_preview = gr.Image(label="Your photo", interactive=False)
                vt_outfit_board = gr.Image(label="Outfit board", interactive=False)

    # Hidden state to hold last matches data
    matches_state = gr.State([])

    # ---------- wiring ----------
    demo.load(load_catalog_all, inputs=None, outputs=[browse_tops, browse_bottoms, browse_dresses, browse_shoes, browse_accessories])

    # populate top_select on load
    def populate_top_dropdown():
        opts = load_top_options()
        # Gradio Dropdown expects list of label or (value, label) tuples depending on version; returning value-label pairs
        return [o for o in opts]

    demo.load(populate_top_dropdown, inputs=None, outputs=[top_select])

    find_matches_btn.click(recommend_matches_ui, inputs=[top_select], outputs=[matches_info, matches_html, matches_state])

    # when matches_state is set, update match_choice dropdown choices via a small helper
    def build_choice_labels(matches):
        if not matches:
            return []
        return [m.get('title') for m in matches]

    demo.load(lambda: build_choice_labels([]), inputs=None, outputs=[match_choice])

    # After find_matches clicked, update match_choice choices via a chained call
    def update_choices_after_find(matches):
        return build_choice_labels(matches)

    find_matches_btn.click(lambda top_id: update_choices_after_find(recommend_matches(top_id, DEMO_USER_ID, limit=3)), inputs=[top_select], outputs=[match_choice])

    save_match_btn.click(add_selected_match_to_saved, inputs=[match_choice, matches_state], outputs=[save_match_msg])

    # Closet add
    def show_closet_html():
        session = session_service.get_session(DEMO_USER_ID)
        items = session.wardrobe
        if not items:
            return "<div style='font-size:13px; opacity:0.7;'>Your closet is empty. Add a few favorites!</div>"
        cards = []
        for w in items:
            colors = ", ".join(w.colors)
            cards.append(f"""
            <div style=\"min-width: 180px; max-width: 220px; background:#ffffff; border-radius:14px; border:1px solid #f3e8ff; box-shadow:0 6px 16px rgba(15,23,42,0.06); padding:10px; display:flex; flex-direction:column; gap:4px;\">
              <div style=\"font-size:12px;font-weight:600;\">{w.name}</div>
              <div style=\"font-size:11px;opacity:0.7;\">Category: {w.category}</div>
              <div style=\"font-size:11px;opacity:0.7;\">Colors: {colors}</div>
              <div style=\"font-size:11px;opacity:0.7;\">{w.notes}</div>
            </div>
            """)
        return f"<div style='overflow-x:auto;padding:4px 0 8px 0;'><div style='display:flex;flex-wrap:nowrap;gap:12px;'>{''.join(cards)}</div></div>"

    demo.load(show_closet_html, inputs=None, outputs=[closet_grid])

    def add_closet_item_ui(name, colors_text, category, notes):
        session = session_service.get_session(DEMO_USER_ID)
        if not name:
            return "Please provide a name.", show_closet_html()
        colors = [c.strip() for c in (colors_text or "").split(",") if c.strip()]
        item = orchestrator.closet.add_item(session, name, colors, category, notes)
        return f"Added '{item.name}' to your closet.", show_closet_html()

    add_btn.click(add_closet_item_ui, inputs=[closet_name, closet_colors, closet_category, closet_notes], outputs=[add_msg, closet_grid])

    # Saved looks
    def load_saved_outfits_ui():
        session = session_service.get_session(DEMO_USER_ID)
        saved_list = session.saved_outfits
        if not saved_list:
            return "<div style='font-size:13px; opacity:0.7;'>You haven't saved any looks yet.</div>"
        cards = []
        for idx, entry in enumerate(saved_list, start=1):
            outfit = entry.get('outfit', {})
            items = entry.get('items', [])
            name = outfit.get('name', f"Look #{idx}")
            total = outfit.get('estimated_price', 0.0)
            count = len(items)
            tags = ", ".join(outfit.get('style_tags', []))
            cards.append(f"""
            <div style=\"min-width: 220px; max-width: 260px; background:#ffffff; border-radius:16px; border:1px solid #f3e8ff; box-shadow:0 6px 16px rgba(15,23,42,0.06); padding:12px; display:flex; flex-direction:column; gap:6px;\">
              <div style=\"font-size:13px;font-weight:650;color:#4b164c;\">{name}</div>
              <div style=\"font-size:11px;opacity:0.85;\">{count} piece(s) · Total ${total:.2f}</div>
              <div style=\"font-size:11px;opacity:0.8;\"><span style=\"font-weight:600;\">Tags:</span> {tags or '—'}</div>
            </div>
            """)
        return f"<div style='overflow-x:auto;padding:4px 0 8px 0;'><div style='display:flex;flex-wrap:nowrap;gap:12px;'>{''.join(cards)}</div></div>"

    demo.load(load_saved_outfits_ui, inputs=None, outputs=[saved_grid])
    refresh_saved_btn.click(load_saved_outfits_ui, inputs=None, outputs=[saved_grid])

    # Trends
    def load_trends_ui():
        trends = trend_agent.get_trends()
        ts = trend_agent.last_refreshed
        info = f"Last refreshed at: {ts}" if ts else "Trends not refreshed yet."
        cards = []
        for t in trends:
            cards.append(f"""
            <div style=\"min-width: 220px; max-width: 260px; background:#ffffff; border-radius:16px; border:1px solid #f3e8ff; box-shadow:0 6px 16px rgba(15,23,42,0.06); padding:12px; display:flex; flex-direction:column; gap:6px;\">
              <div style=\"font-size:13px;font-weight:650;color:#4b164c;\">{t['name']}</div>
              <div style=\"font-size:11px;opacity:0.85;\">{t['vibe']}</div>
              <div style=\"font-size:11px;opacity:0.8;\"><span style=\"font-weight:600;\">Tags:</span> {', '.join(t['tags'])}</div>
            </div>
            """)
        html = f"<div style='overflow-x:auto;padding:4px 0 8px 0;'><div style='display:flex;gap:12px;'>{''.join(cards)}</div></div>"
        return info, html

    demo.load(load_trends_ui, inputs=None, outputs=[trends_info, trends_grid])
    refresh_btn.click(lambda: load_trends_ui(), inputs=None, outputs=[trends_info, trends_grid])

    # Virtual try-on simple wiring
    def get_last_outfit_board(user_id: str = DEMO_USER_ID):
        session = session_service.get_session(user_id)
        if not session.last_outfits:
            return None, "No outfit yet. Ask the stylist first."
        last_outfit = session.last_outfits[-1]
        items = [product_by_id(pid).__dict__ for pid in last_outfit['item_ids'] if product_by_id(pid)]
        board = build_outfit_board_image(items)
        if board is None:
            return None, "Could not build outfit board."
        return board, f"Using your last outfit: {last_outfit.get('name', 'Styled Look')}"

    def virtual_try_on(user_photo):
        if user_photo is None:
            return None, None, "Upload a photo first."
        board, info = get_last_outfit_board()
        return user_photo, board, info

    try_btn.click(virtual_try_on, inputs=[user_photo], outputs=[vt_user_preview, vt_outfit_board, vt_caption])

    demo.launch()

# End of file


⚠️ No GEMINI_API_KEY found. StylistAgent will use a rule-based fallback.
Loaded 16 demo products.
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7545637838478cc4c1.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# StyleSphere AI — Enhanced Boutique Demo (v2)
# Integrates: recommend_matches, shopping cart, improved images, size advisor, styling/UI tweaks,
# semantic search (keyword-based), and an in-app analytics dashboard.
# Single-file Gradio app. Install: pip install google-generativeai gradio pillow requests

import os
import json
import random
import time
from datetime import datetime
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple

import requests
from io import BytesIO
from PIL import Image, ImageOps

import google.generativeai as genai
import gradio as gr

# ---------------- Config ----------------
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", None)
if GEMINI_API_KEY:
    genai.configure(api_key=GEMINI_API_KEY)
    print("✅ Gemini configured from environment variable.")
else:
    print("⚠️ No GEMINI_API_KEY found. StylistAgent will use a rule-based fallback.")

# ---------------- Logging / Analytics (in-memory) ----------------
ANALYTICS_LOGS: List[dict] = []

def log_event(agent_name: str, event: str, data: Optional[dict] = None):
    payload = {"time": datetime.utcnow().isoformat() + "Z", "agent": agent_name, "event": event, "data": data or {}}
    ANALYTICS_LOGS.append(payload)
    print(f"[LOG] {json.dumps(payload)}")

# ---------------- Data models & demo catalog ----------------
@dataclass
class Product:
    id: str
    name: str
    brand: str
    price: float
    image_url: str
    colors: List[str]
    categories: List[str]
    sizes: List[str] = field(default_factory=lambda: ["XS","S","M","L","XL"])  # demo

PRODUCTS = [
    Product(id="1", name="Pastel Linen Shirt", brand="Breeze", price=40.0,
            image_url="https://via.placeholder.com/600x600.png?text=Pastel+Linen+Shirt", colors=["pastel-pink","white"], categories=["top","smart-casual","pastel"]),
    Product(id="2", name="Cream Knit Sweater", brand="CozyHome", price=55.0,
            image_url="https://via.placeholder.com/600x600.png?text=Cream+Knit+Sweater", colors=["cream"], categories=["top","casual","neutral"]),
    Product(id="3", name="White Cotton Tee", brand="Everyday", price=20.0,
            image_url="https://via.placeholder.com/600x600.png?text=White+Cotton+Tee", colors=["white"], categories=["top","casual","basic"]),
    Product(id="4", name="Soft Blue Blouse", brand="Skyline", price=45.0,
            image_url="https://via.placeholder.com/600x600.png?text=Soft+Blue+Blouse", colors=["pastel-blue"], categories=["top","smart-casual","pastel"]),
    Product(id="5", name="White Chino Pants", brand="UrbanEase", price=45.0,
            image_url="https://via.placeholder.com/600x600.png?text=White+Chinos", colors=["white"], categories=["bottom","smart-casual","neutral"]),
    Product(id="6", name="Beige Wide-Leg Trousers", brand="MinimalCo", price=60.0,
            image_url="https://via.placeholder.com/600x600.png?text=Beige+Trousers", colors=["beige"], categories=["bottom","formal","minimal"]),
    Product(id="7", name="Light Wash Mom Jeans", brand="DenimDays", price=50.0,
            image_url="https://via.placeholder.com/600x600.png?text=Mom+Jeans", colors=["light-blue"], categories=["bottom","casual"]),
    Product(id="8", name="Pastel Maxi Dress", brand="Sunset Bloom", price=70.0,
            image_url="https://via.placeholder.com/600x600.png?text=Pastel+Maxi+Dress", colors=["pastel-blue","pastel-pink"], categories=["dress","occasion","wedding","pastel"]),
    Product(id="9", name="Satin Slip Dress", brand="GlowWear", price=75.0,
            image_url="https://via.placeholder.com/600x600.png?text=Satin+Slip+Dress", colors=["champagne"], categories=["dress","evening","date"]),
    Product(id="10", name="Floral Day Dress", brand="Gardenia", price=55.0,
            image_url="https://via.placeholder.com/600x600.png?text=Floral+Day+Dress", colors=["white","pastel-pink"], categories=["dress","casual","day"]),
    Product(id="11", name="Tan Loafers", brand="StepRight", price=50.0,
            image_url="https://via.placeholder.com/600x600.png?text=Tan+Loafers", colors=["tan"], categories=["shoes","smart-casual"]),
    Product(id="12", name="Strappy Sandals", brand="SoleWave", price=35.0,
            image_url="https://via.placeholder.com/600x600.png?text=Strappy+Sandals", colors=["beige"], categories=["shoes","occasion","summer"]),
    Product(id="13", name="White Sneakers", brand="StreetStep", price=45.0,
            image_url="https://via.placeholder.com/600x600.png?text=White+Sneakers", colors=["white"], categories=["shoes","casual"]),
    Product(id="14", name="Statement Pearl Earrings", brand="Aurora", price=20.0,
            image_url="https://via.placeholder.com/600x600.png?text=Pearl+Earrings", colors=["white"], categories=["accessory","occasion"]),
    Product(id="15", name="Cream Tote Bag", brand="DailyCarry", price=40.0,
            image_url="https://via.placeholder.com/600x600.png?text=Cream+Tote+Bag", colors=["cream"], categories=["accessory","casual","minimal"]),
    Product(id="16", name="Gold Layered Necklace", brand="GlowWear", price=22.0,
            image_url="https://via.placeholder.com/600x600.png?text=Gold+Necklace", colors=["gold"], categories=["accessory"]),
]

# ---------------- Utilities ----------------
def product_by_id(pid: str) -> Optional[Product]:
    for p in PRODUCTS:
        if p.id == str(pid):
            return p
    return None


def safe_fetch_image(url: str, size: Tuple[int,int]=(400,400)):
    """Fetch image and return a PIL Image; on failure return a neutral placeholder image."""
    try:
        resp = requests.get(url, timeout=4)
        img = Image.open(BytesIO(resp.content)).convert("RGB")
        img = ImageOps.fit(img, size, Image.LANCZOS)
        return img
    except Exception as e:
        log_event("Image", "fetch_error", {"url": url, "error": str(e)})
        # generate a simple placeholder
        placeholder = Image.new("RGB", size, (250,250,253))
        return placeholder


def product_search(max_price: Optional[float] = None, min_price: Optional[float] = None,
                   colors: Optional[List[str]] = None, categories: Optional[List[str]] = None, text_query: Optional[str] = None, limit: int = 20) -> List[Product]:
    results = []
    q = (text_query or "").lower().strip()
    for p in PRODUCTS:
        if max_price is not None and p.price > max_price:
            continue
        if min_price is not None and p.price < min_price:
            continue
        if colors and not any(c in p.colors for c in colors):
            continue
        if categories and not any(cat in p.categories for cat in categories):
            continue
        # simple semantic-ish match: check words in name, brand, categories
        if q:
            score = 0
            for tok in q.split():
                if tok in p.name.lower() or tok in p.brand.lower():
                    score += 2
                if tok in " ".join(p.categories):
                    score += 1
                if tok in ",".join(p.colors):
                    score += 1
            if score == 0:
                continue
        results.append(p)
    log_event("product_search", "filtered", {"query": text_query, "count": len(results)})
    return results[:limit]


def color_harmony_score(items: List[Product]) -> float:
    if not items:
        return 0.0
    score = 0
    for p in items:
        for c in p.colors:
            if "pastel" in c or c in ["white", "beige", "tan", "cream"]:
                score += 2
            else:
                score += 1
    return min(1.0, score / (len(items) * 3))

# ---------------- Session & simple persistent store ----------------
@dataclass
class UserPreferences:
    budget: Optional[float] = None
    styles: List[str] = field(default_factory=list)
    disliked_colors: List[str] = field(default_factory=list)
    preferred_fit: Optional[str] = None

@dataclass
class WardrobeItem:
    id: str
    name: str
    colors: List[str]
    category: str
    notes: str = ""

@dataclass
class SessionState:
    user_id: str
    preferences: UserPreferences = field(default_factory=UserPreferences)
    wardrobe: List[WardrobeItem] = field(default_factory=list)
    last_outfits: List[dict] = field(default_factory=list)
    saved_outfits: List[dict] = field(default_factory=list)
    cart: List[dict] = field(default_factory=list)  # items with qty & size

class SessionService:
    def __init__(self):
        self.sessions: Dict[str, SessionState] = {}
    def get_session(self, user_id: str) -> SessionState:
        if user_id not in self.sessions:
            self.sessions[user_id] = SessionState(user_id=user_id)
            log_event("SessionService", "create_session", {"user_id": user_id})
        return self.sessions[user_id]

session_service = SessionService()
DEMO_USER_ID = "demo_user"

# ---------------- Agents (stylist fallback) ----------------
class StylistAgent:
    def __init__(self):
        self.model = genai.GenerativeModel("gemini-1.5-pro") if GEMINI_API_KEY else None
    def create_outfit(self, user_message: str, session: SessionState, candidate_products: List[Product]) -> dict:
        log_event("StylistAgent", "create_outfit_start", {"msg": user_message})
        return self._fallback_outfit(user_message, session, candidate_products)
    def _fallback_outfit(self, user_message: str, session: SessionState, candidate_products: List[Product]) -> dict:
        items = []
        lower = user_message.lower()
        if "wedding" in lower:
            dresses = [p for p in candidate_products if "dress" in p.categories]
            shoes = [p for p in candidate_products if "shoes" in p.categories]
            accessories = [p for p in candidate_products if "accessory" in p.categories]
            if dresses:
                items.append(random.choice(dresses).id)
            if shoes:
                items.append(random.choice(shoes).id)
            if accessories:
                items.append(random.choice(accessories).id)
        else:
            tops = [p for p in candidate_products if "top" in p.categories]
            bottoms = [p for p in candidate_products if "bottom" in p.categories]
            shoes = [p for p in candidate_products if "shoes" in p.categories]
            if tops:
                items.append(random.choice(tops).id)
            if bottoms:
                items.append(random.choice(bottoms).id)
            if shoes:
                items.append(random.choice(shoes).id)
        total_price = sum(product_by_id(pid).price for pid in items)
        return {"name": "Simple Styled Look", "description": "A simple fallback outfit.", "item_ids": items, "style_tags": ["fallback"], "estimated_price": total_price}

stylist_agent = StylistAgent()

# ---------------- Budget & closet agents ----------------
class BudgetAgent:
    def optimize_outfit(self, outfit: dict, session: SessionState) -> dict:
        budget = session.preferences.budget
        if budget is None:
            return outfit
        ids = outfit["item_ids"]
        items = [product_by_id(pid) for pid in ids if product_by_id(pid)]
        total = sum(p.price for p in items)
        if total <= budget:
            outfit["budget_status"] = f"✅ Within budget (${total:.2f} of ${budget:.2f})"
            return outfit
        items_sorted = sorted(items, key=lambda p: p.price, reverse=True)
        for expensive in items_sorted:
            candidates = product_search(max_price=expensive.price - 10, categories=expensive.categories, limit=3)
            if not candidates:
                continue
            cheaper = candidates[0]
            outfit["item_ids"].remove(expensive.id)
            outfit["item_ids"].append(cheaper.id)
            break
        new_items = [product_by_id(pid) for pid in outfit["item_ids"] if product_by_id(pid)]
        total_new = sum(p.price for p in new_items)
        status = "✅" if total_new <= budget else "⚠️"
        outfit["estimated_price"] = total_new
        outfit["budget_status"] = f"{status} Adjusted price: ${total_new:.2f} (budget ${budget:.2f})"
        return outfit

budget_agent = BudgetAgent()

class ClosetAgent:
    def add_item(self, session: SessionState, name: str, colors: List[str], category: str, notes: str = ""):
        wid = f"w{len(session.wardrobe)+1}"
        item = WardrobeItem(id=wid, name=name, colors=colors, category=category, notes=notes)
        session.wardrobe.append(item)
        log_event("ClosetAgent", "add_item", {"id": wid})
        return item

closet_agent = ClosetAgent()

# ---------------- Trend agent ----------------
class TrendAgent:
    def __init__(self):
        self.trends = []
        self.last_refreshed = None
    def refresh_trends(self):
        log_event("TrendAgent", "refresh_start", {})
        time.sleep(0.2)
        now = datetime.utcnow().isoformat() + "Z"
        self.last_refreshed = now
        self.trends = [
            {"name": "Soft Pastel Wedding Guest", "vibe": "Light, romantic, flowy silhouettes in pastel tones.", "tags": ["pastel","wedding","romantic"]},
            {"name": "Minimal Resort Linen", "vibe": "Crisp whites and linens, relaxed tailoring, beach-perfect.", "tags": ["minimal","linen","resort"]},
            {"name": "Statement Accessories", "vibe": "Clean base outfits with bold earrings and bags.", "tags": ["accessories","statement","elevated-basics"]},
        ]
        log_event("TrendAgent", "refresh_done", {"count": len(self.trends)})
        return self.trends, now
    def get_trends(self):
        if not self.trends:
            trends, _ = self.refresh_trends()
            return trends
        return self.trends

trend_agent = TrendAgent()

# ---------------- Evaluation & UI helpers ----------------
COLOR_MAP = {"pastel-pink":"#ffc1cc","pastel-blue":"#c4e1ff","white":"#ffffff","cream":"#f5f5e8","beige":"#f5f5dc","tan":"#d2b48c","light-blue":"#cde4ff","gold":"#f3c969","default":"#cccccc"}

def color_to_hex(c: str) -> str:
    return COLOR_MAP.get(c.lower(), COLOR_MAP["default"])


def build_products_html(items: list, show_match_buttons: bool = False) -> str:
    if not items:
        return "<div style='font-size:13px; opacity:0.7;'>No products to show.</div>"
    cards = []
    for p in items:
        img_url = p.get("image_url", "")
        name = p.get("name", "Item")
        brand = p.get("brand", "")
        price = p.get("price", 0)
        colors = ", ".join(p.get("colors", []))
        match_btn = "<div style='font-size:12px;opacity:0.85;margin-top:6px;'>Click for matches</div>" if show_match_buttons else ""
        cards.append(f"""
        <div style="min-width: 180px; max-width: 220px; background:#ffffff; border-radius:14px; border:1px solid #f3e8ff; box-shadow:0 6px 16px rgba(15,23,42,0.06); padding:10px; display:flex; flex-direction:column; gap:6px;">
          <div style="width:100%;border-radius:10px;overflow:hidden;background:#f9fafb;"><img src=\"{img_url}\" alt=\"{name}\" style=\"width:100%;height:180px;object-fit:cover;display:block;\" /></div>
          <div style="font-size:12px;font-weight:600;line-height:1.2;">{name}</div>
          <div style="font-size:11px;opacity:0.7;">{brand}</div>
          <div style="font-size:12px;font-weight:600;color:#4b164c;">${price:.2f}</div>
          <div style="font-size:11px;opacity:0.7;">{colors}</div>
          {match_btn}
        </div>
        """)
    return f"<div style='overflow-x:auto;padding:4px 0 8px 0;'><div style='display:flex;flex-wrap:nowrap;gap:12px;'>{''.join(cards)}</div></div>"


def build_outfit_board_image(items: list):
    if not items:
        return None
    try:
        imgs = []
        target_h = 220
        for p in items:
            url = p.get("image_url")
            if not url:
                continue
            resp = requests.get(url, timeout=5)
            img = Image.open(BytesIO(resp.content)).convert("RGB")
            w, h = img.size
            new_w = int(w * (target_h / h))
            imgs.append(img.resize((new_w, target_h)))
        if not imgs:
            return None
        total_w = sum(img.size[0] for img in imgs)
        board = Image.new("RGB", (total_w, target_h), (250, 250, 255))
        x = 0
        for img in imgs:
            board.paste(img, (x, 0))
            x += img.size[0]
        return board
    except Exception as e:
        log_event("UI", "board_image_error", {"error": str(e)})
        return None

# ---------------- Closet bias ----------------
def apply_closet_bias(session: SessionState, candidates: List[Product]) -> List[Product]:
    closet_colors = set()
    for item in session.wardrobe:
        closet_colors.update(item.colors)
    if not closet_colors:
        return candidates
    return sorted(candidates, key=lambda p: len(closet_colors & set(p.colors)), reverse=True)

# ---------------- Recommend matches ----------------
def score_candidate_combo(top: Product, bottom: Product, shoe: Product, accessory: Product, session: SessionState):
    items = [top, bottom, shoe, accessory]
    color_score = color_harmony_score(items)
    top_cats = set(top.categories)
    common = 0
    for other in (bottom, shoe, accessory):
        if top_cats & set(other.categories):
            common += 1
    category_score = common / 3.0
    total = sum(p.price for p in items)
    target = top.price * 2.0
    price_diff = abs(total - target) / max(target, 1.0)
    price_score = max(0.0, 1.0 - price_diff)
    score = 0.5 * color_score + 0.3 * category_score + 0.2 * price_score
    explain = f"{top.name} pairs with {bottom.name}, {shoe.name} and {accessory.name} — color harmony {color_score:.2f}, vibe match {category_score:.2f}."
    return score, explain, total


def recommend_matches(product_id: str, user_id: str = DEMO_USER_ID, limit: int = 3):
    session = session_service.get_session(user_id)
    top = product_by_id(product_id)
    if not top:
        return []
    bottoms = [p for p in PRODUCTS if "bottom" in p.categories]
    shoes = [p for p in PRODUCTS if "shoes" in p.categories]
    accessories = [p for p in PRODUCTS if "accessory" in p.categories]
    bottoms = apply_closet_bias(session, bottoms)
    shoes = apply_closet_bias(session, shoes)
    accessories = apply_closet_bias(session, accessories)
    combos = []
    for b in bottoms[:6]:
        for s in shoes[:6]:
            for a in accessories[:6]:
                score, explain, total = score_candidate_combo(top, b, s, a, session)
                combos.append({"score": score, "explain": explain, "estimated_price": total, "items": [b.id, s.id, a.id], "items_full": [b.__dict__, s.__dict__, a.__dict__]})
    combos_sorted = sorted(combos, key=lambda c: c["score"], reverse=True)[:limit]
    results = []
    for idx, c in enumerate(combos_sorted, start=1):
        title = f"Match #{idx}"
        one_liner = f"{top.name} + {c['items_full'][0]['name']} + {c['items_full'][1]['name']} — {c['explain'].split('—')[-1].strip()}"
        results.append({"title": title, "one_liner": one_liner, "item_ids": [top.id] + c["items"], "estimated_price": c["estimated_price"], "score": c["score"], "images": [top.image_url] + [it["image_url"] for it in c["items_full"]], "items_full": c["items_full"]})
    log_event("recommend_matches", "generated", {"top": top.id, "user": user_id, "count": len(results)})
    return results

# ---------------- Shopping cart ----------------
def add_to_cart(user_id: str, product_id: str, qty: int = 1, size: Optional[str] = None):
    session = session_service.get_session(user_id)
    p = product_by_id(product_id)
    if not p:
        return "Product not found."
    # if same item+size exists, increment
    for it in session.cart:
        if it.get("product_id") == p.id and it.get("size") == size:
            it["qty"] += qty
            log_event("Cart", "update_qty", {"user": user_id, "product": p.id, "qty": it["qty"], "size": size})
            return "Updated cart."
    session.cart.append({"product_id": p.id, "qty": qty, "size": size})
    log_event("Cart", "add", {"user": user_id, "product": p.id, "qty": qty, "size": size})
    return "Added to cart."


def cart_summary(user_id: str):
    session = session_service.get_session(user_id)
    items = []
    total = 0.0
    for it in session.cart:
        p = product_by_id(it["product_id"])
        if not p:
            continue
        subtotal = p.price * it["qty"]
        items.append({"id": p.id, "name": p.name, "brand": p.brand, "price": p.price, "qty": it["qty"], "size": it.get("size"), "subtotal": subtotal})
        total += subtotal
    return items, total


def remove_from_cart(user_id: str, product_id: str, size: Optional[str] = None):
    session = session_service.get_session(user_id)
    before = len(session.cart)
    session.cart = [it for it in session.cart if not (it["product_id"] == product_id and it.get("size") == size)]
    after = len(session.cart)
    log_event("Cart", "remove", {"user": user_id, "product": product_id, "size": size, "removed": before - after})
    return "Removed item(s) from cart."

# ---------------- Size advisor (simple heuristic) ----------------
def size_advice_for_product(product: Product, height_cm: int, weight_kg: int, usual_size: Optional[str] = None) -> str:
    # naive mapping just for demo purposes
    bmi = weight_kg / ((height_cm/100) ** 2) if height_cm and weight_kg else 22
    # pick size by BMI thresholds (toy example)
    if bmi < 19:
        size = "S"
    elif bmi < 24:
        size = "M"
    elif bmi < 29:
        size = "L"
    else:
        size = "XL"
    # prefer usual_size if provided and exists in product
    if usual_size and usual_size in product.sizes:
        preferred = usual_size
    else:
        preferred = size if size in product.sizes else product.sizes[min(len(product.sizes)-1, 2)]
    advice = f"Based on height {height_cm}cm and weight {weight_kg}kg, we suggest size **{preferred}** (approx. BMI {bmi:.1f})."
    log_event("SizeAdvisor", "advice", {"product": product.id, "height": height_cm, "weight": weight_kg, "advice_size": preferred})
    return advice

# ---------------- Semantic-ish search helper ----------------
SYNONYMS = {
    "flowy": ["flowy","flowing","flowy"],
    "pastel": ["pastel","soft","light"],
    "linen": ["linen"],
    "casual": ["casual","everyday","day"],
}

def expand_query_terms(q: str) -> List[str]:
    terms = q.lower().split()
    expanded = set(terms)
    for t in terms:
        for k, vals in SYNONYMS.items():
            if t == k or t in vals:
                expanded.update(vals)
    return list(expanded)

# ---------------- Recommend matches UI helpers ----------------
def build_match_cards_html(matches: list) -> str:
    if not matches:
        return "<div style='font-size:13px; opacity:0.7;'>No matches found.</div>"
    cards = []
    for m in matches:
        imgs_html = "".join([f"<img src='{u}' style='width:80px;height:80px;object-fit:cover;border-radius:8px;margin-right:6px;' />" for u in m['images']])
        cards.append(f"""
        <div style="min-width:240px;max-width:280px;background:#fff;border-radius:14px;border:1px solid #f3e8ff;padding:12px;display:flex;flex-direction:column;gap:8px;">
          <div style="font-weight:650;color:#4b164c;">{m['title']}</div>
          <div style="font-size:12px;opacity:0.9;">{m['one_liner']}</div>
          <div style="display:flex;align-items:center;margin-top:6px;">{imgs_html}</div>
          <div style="display:flex;justify-content:space-between;align-items:center;margin-top:8px;">
            <div style="font-weight:600;color:#4b164c;">${m['estimated_price']:.2f}</div>
            <div style="font-size:12px;opacity:0.85;">Score: {m['score']:.2f}</div>
          </div>
        </div>
        """)
    return f"<div style='overflow-x:auto;padding:4px 0 8px 0;'><div style='display:flex;gap:12px;'>{''.join(cards)}</div></div>"

# ---------------- Saved outfits ----------------

def save_match_as_outfit(user_id: str, match: dict):
    session = session_service.get_session(user_id)
    items = [product_by_id(pid).__dict__ for pid in match["item_ids"] if product_by_id(pid)]
    session.saved_outfits.append({"outfit": {"name": match.get("title", "Matched Look"), "item_ids": match["item_ids"], "estimated_price": match.get("estimated_price", 0.0), "style_tags": []}, "items": items})
    log_event("SavedMatch", "save", {"user": user_id, "title": match.get("title")})
    return f"Saved '{match.get('title', 'Look')}' to your Saved Looks."

# ---------------- Orchestrator simplified ----------------
class Orchestrator:
    def handle_message(self, user_id: str, message: str) -> dict:
        session = session_service.get_session(user_id)
        # detect budget in message (simple)
        words = message.lower().replace("$", "").split()
        nums = [w for w in words if w.replace('.', '', 1).isdigit()]
        if nums:
            try:
                session.preferences.budget = float(nums[0])
            except:
                pass
        candidates = product_search(max_price=session.preferences.budget, limit=20) if session.preferences.budget else PRODUCTS
        candidates = apply_closet_bias(session, candidates)
        outfit = stylist_agent.create_outfit(message, session, candidates)
        outfit = budget_agent.optimize_outfit(outfit, session)
        items = [product_by_id(pid).__dict__ for pid in outfit.get("item_ids", []) if product_by_id(pid)]
        outfit["scores"] = {"overall": 0.8, "color_harmony": 0.9}
        session.last_outfits.append(outfit)
        log_event("Orchestrator", "outfit_generated", {"user": user_id, "name": outfit.get('name')})
        return {"type": "outfit", "outfit": outfit, "items": items}

orchestrator = Orchestrator()

# ---------------- Gradio UI ----------------

theme = gr.themes.Soft(primary_hue="pink", secondary_hue="indigo", radius_size="lg").set(
    body_background_fill="#faf5ff",
    body_text_color="#111827",
    block_background_fill="#ffffff",
    block_border_width="1px",
    block_border_color="#f3e8ff",
)

with gr.Blocks(title="StyleSphere AI — Enhanced Boutique", theme=theme) as demo:
    gr.Markdown("""
# 🌸 StyleSphere AI — Enhanced Boutique
Now with: cart, size advisor, semantic-ish search, improved images, analytics, and match-this-top flow.
""")

    with gr.Tab("Shop"):
        with gr.Row():
            with gr.Column(scale=3):
                search_box = gr.Textbox(label="Search (try: 'flowy pastel top')", placeholder="Search products or describe a look")
                search_btn = gr.Button("Search")
                results_html = gr.HTML(label="Results")
                # catalog sections
                gr.Markdown("**Catalog**")
                catalog_all_html = gr.HTML(label="All products")
            with gr.Column(scale=1):
                gr.Markdown("### 🛒 Cart & Size Advisor")
                cart_md = gr.Markdown(label="Cart")
                checkout_btn = gr.Button("Checkout (simulate)")
                gr.Markdown("---")
                gr.Markdown("### Size Advisor")
                sa_product = gr.Dropdown(choices=[], label="Product for size advice")
                sa_height = gr.Number(label="Height (cm)", value=170)
                sa_weight = gr.Number(label="Weight (kg)", value=65)
                sa_usize = gr.Dropdown(choices=["","XS","S","M","L","XL"], label="Your usual size (optional)")
                sa_btn = gr.Button("Get size advice")
                sa_out = gr.Markdown()

        # match flow quick access
        with gr.Row():
            with gr.Column(scale=2):
                top_select = gr.Dropdown(choices=[], label="Select a top to find matches")
                find_matches_btn = gr.Button("Find matches for this top")
            with gr.Column(scale=2):
                matches_html = gr.HTML(label="Matches")

    with gr.Tab("Stylist Chat"):
        with gr.Row():
            with gr.Column(scale=3):
                chat_input = gr.Textbox(label="Tell the stylist what you need", placeholder="e.g. outfit for office meeting under $100 in neutral tones")
                chat_send = gr.Button("Ask stylist")
                chat_out = gr.Markdown()
            with gr.Column(scale=1):
                gr.Markdown("### Quick Actions")
                quick_top = gr.Dropdown(choices=[], label="Quick top")
                quick_match = gr.Button("Find matches for quick top")

    with gr.Tab("My Closet"):
        with gr.Row():
            with gr.Column(scale=1):
                cname = gr.Textbox(label="Item name")
                ccolors = gr.Textbox(label="Colors (comma-separated)")
                ccat = gr.Dropdown(["top","bottom","dress","shoes","accessory","outerwear"], label="Category", value="top")
                cnotes = gr.Textbox(label="Notes")
                add_closet_btn = gr.Button("Add to Closet")
                add_closet_msg = gr.Markdown()
            with gr.Column(scale=2):
                closet_html = gr.HTML()

    with gr.Tab("Saved Looks"):
        saved_html = gr.HTML()
        refresh_saved_btn = gr.Button("Refresh saved looks")

    with gr.Tab("Trends & Analytics"):
        with gr.Row():
            with gr.Column():
                trends_info = gr.Markdown()
                trends_html = gr.HTML()
            with gr.Column():
                gr.Markdown("### Admin Analytics (in-memory)")
                logs_json = gr.Textbox(label="Recent logs (JSON)")
                clear_logs_btn = gr.Button("Clear logs")

    # Hidden state
    matches_state = gr.State([])

    # ---------- wiring ----------
    def load_catalog_all():
        items = [p.__dict__ for p in PRODUCTS]
        return build_products_html(items, show_match_buttons=True)

    demo.load(load_catalog_all, inputs=None, outputs=[catalog_all_html])

    def populate_product_dropdowns():
        opts = [(p.id, f"{p.name} — ${p.price:.2f}") for p in PRODUCTS if "top" in p.categories]
        # Gradio Dropdown accepts list of strings or tuples depending on version; return list of (value, label) tuples
        return [o for o in opts]

    demo.load(populate_product_dropdowns, inputs=None, outputs=[top_select, quick_top])

    # Search
    def on_search(q: str):
        if not q:
            items = [p.__dict__ for p in PRODUCTS]
            return build_products_html(items)
        expanded = expand_query_terms(q)
        # do multiple queries and union results
        found = []
        for term in expanded:
            found += product_search(text_query=term, limit=20)
        # dedupe
        seen = set()
        uniq = []
        for p in found:
            if p.id not in seen:
                seen.add(p.id)
                uniq.append(p.__dict__)
        log_event("Search", "query", {"query": q, "expanded": expanded, "found": len(uniq)})
        return build_products_html(uniq)

    search_btn.click(on_search, inputs=[search_box], outputs=[results_html])

    # Size advisor
    def load_sa_product_choices():
        return [(p.id, f"{p.name} — ${p.price:.2f}") for p in PRODUCTS]

    demo.load(load_sa_product_choices, inputs=None, outputs=[sa_product])

    def on_size_advice(prod_choice, h, w, usual):
        if not prod_choice:
            return "Please select a product."
        p = product_by_id(prod_choice)
        if not p:
            return "Product not found."
        advice = size_advice_for_product(p, int(h or 0), int(w or 0), usual)
        return advice

    sa_btn.click(on_size_advice, inputs=[sa_product, sa_height, sa_weight, sa_usize], outputs=[sa_out])

    # Match flow wiring
    def on_find_matches(top_id):
        if not top_id:
            return "Select a top.", "<div style='font-size:13px; opacity:0.7;'>No matches yet.</div>", []
        matches = recommend_matches(top_id, DEMO_USER_ID, limit=3)
        html = build_match_cards_html(matches)
        return f"Showing {len(matches)} matches.", html, matches

    find_matches_btn.click(on_find_matches, inputs=[top_select], outputs=[gr.Markdown(), matches_html, matches_state])

    # Cart wiring: add buttons in product cards are static HTML (not interactive). Provide explicit add-to-cart helper UI for demo
    def add_to_cart_ui(product_id: str, qty: int, size: str):
        if not product_id:
            return "Select a product id to add.", *cart_summary_ui(DEMO_USER_ID)
        msg = add_to_cart(DEMO_USER_ID, product_id, qty or 1, size if size else None)
        return msg, *cart_summary_ui(DEMO_USER_ID)

    def cart_summary_ui(user_id: str):
        session = session_service.get_session(user_id)
        items = []
        total = 0.0
        for it in session.cart:
            p = product_by_id(it["product_id"])
            if not p:
                continue
            subtotal = p.price * it["qty"]
            items.append({"id": p.id, "name": p.name, "brand": p.brand, "price": p.price, "qty": it["qty"], "size": it.get("size"), "subtotal": subtotal})
            total += subtotal
        return "\n".join(lines), items, total

    # wire a small add-to-cart form (product id input) to demonstrate cart
    add_cart_pid = gr.Dropdown(choices=[(p.id, f"{p.name} — ${p.price:.2f}") for p in PRODUCTS], label="Add product to cart")
    add_cart_qty = gr.Slider(minimum=1, maximum=5, step=1, value=1, label="Qty")
    add_cart_size = gr.Dropdown(choices=["","XS","S","M","L","XL"], label="Size (optional)")
    add_cart_btn = gr.Button("Add to cart")

    def add_cart_click(pid, qty, size):
        msg = add_to_cart(DEMO_USER_ID, pid, int(qty or 1), size if size else None)
        log_event("UI", "add_cart_click", {"product": pid, "qty": qty, "size": size})
        cart_text, _, _ = cart_summary_ui(DEMO_USER_ID)
        return msg, cart_text

    add_cart_btn.click(add_cart_click, inputs=[add_cart_pid, add_cart_qty, add_cart_size], outputs=[gr.Markdown(), cart_md])

    checkout_btn.click(lambda: (log_event("Cart","checkout",{"user":DEMO_USER_ID}), "Checkout simulated — thanks!"), inputs=None, outputs=[gr.Markdown()])

    # Closet add
    def show_closet_html():
        session = session_service.get_session(DEMO_USER_ID)
        items = session.wardrobe
        if not items:
            return "<div style='font-size:13px; opacity:0.7;'>Your closet is empty. Add a few favorites!</div>"
        cards = []
        for w in items:
            colors = ", ".join(w.colors)
            cards.append(f"""
            <div style="min-width: 180px; max-width: 220px; background:#ffffff; border-radius:14px; border:1px solid #f3e8ff; box-shadow:0 6px 16px rgba(15,23,42,0.06); padding:10px; display:flex; flex-direction:column; gap:4px;">
              <div style="font-size:12px;font-weight:600;">{w.name}</div>
              <div style="font-size:11px;opacity:0.7;">Category: {w.category}</div>
              <div style="font-size:11px;opacity:0.7;">Colors: {colors}</div>
              <div style="font-size:11px;opacity:0.7;">{w.notes}</div>
            </div>
            """)
        return f"<div style='overflow-x:auto;padding:4px 0 8px 0;'><div style='display:flex;flex-wrap:nowrap;gap:12px;'>{''.join(cards)}</div></div>"

    demo.load(show_closet_html, inputs=None, outputs=[closet_html])

    def add_closet_item_ui(name, colors_text, category, notes):
        session = session_service.get_session(DEMO_USER_ID)
        if not name:
            return "Please provide a name.", show_closet_html()
        colors = [c.strip() for c in (colors_text or "").split(",") if c.strip()]
        item = closet_agent.add_item(session, name, colors, category, notes)
        return f"Added '{item.name}' to your closet.", show_closet_html()

    add_closet_btn.click(add_closet_item_ui, inputs=[cname, ccolors, ccat, cnotes], outputs=[add_closet_msg, closet_html])

    # Saved looks
    def load_saved_outfits_ui():
        session = session_service.get_session(DEMO_USER_ID)
        saved_list = session.saved_outfits
        if not saved_list:
            return "<div style='font-size:13px; opacity:0.7;'>You haven't saved any looks yet.</div>"
        cards = []
        for idx, entry in enumerate(saved_list, start=1):
            outfit = entry.get('outfit', {})
            items = entry.get('items', [])
            name = outfit.get('name', f"Look #{idx}")
            total = outfit.get('estimated_price', 0.0)
            count = len(items)
            tags = ", ".join(outfit.get('style_tags', []))
            cards.append(f"""
            <div style="min-width: 220px; max-width: 260px; background:#ffffff; border-radius:16px; border:1px solid #f3e8ff; box-shadow:0 6px 16px rgba(15,23,42,0.06); padding:12px; display:flex; flex-direction:column; gap:6px;">
              <div style="font-size:13px;font-weight:650;color:#4b164c;">{name}</div>
              <div style="font-size:11px;opacity:0.85;">{count} piece(s) · Total ${total:.2f}</div>
              <div style="font-size:11px;opacity:0.8;"><span style="font-weight:600;">Tags:</span> {tags or '—'}</div>
            </div>
            """)
        return f"<div style='overflow-x:auto;padding:4px 0 8px 0;'><div style='display:flex;flex-wrap:nowrap;gap:12px;'>{''.join(cards)}</div></div>"

    demo.load(load_saved_outfits_ui, inputs=None, outputs=[saved_html])
    refresh_saved_btn.click(load_saved_outfits_ui, inputs=None, outputs=[saved_html])

    # Trends & analytics
    def load_trends_ui():
        trends = trend_agent.get_trends()
        ts = trend_agent.last_refreshed
        info = f"Last refreshed at: {ts}" if ts else "Trends not refreshed yet."
        cards = []
        for t in trends:
            cards.append(f"""
            <div style="min-width: 220px; max-width: 260px; background:#ffffff; border-radius:16px; border:1px solid #f3e8ff; box-shadow:0 6px 16px rgba(15,23,42,0.06); padding:12px; display:flex; flex-direction:column; gap:6px;">
              <div style="font-size:13px;font-weight:650;color:#4b164c;">{t['name']}</div>
              <div style="font-size:11px;opacity:0.85;">{t['vibe']}</div>
              <div style="font-size:11px;opacity:0.8;"><span style="font-weight:600;">Tags:</span> {', '.join(t['tags'])}</div>
            </div>
            """)
        html = f"<div style='overflow-x:auto;padding:4px 0 8px 0;'><div style='display:flex;gap:12px;'>{''.join(cards)}</div></div>"
        return info, html

    demo.load(load_trends_ui, inputs=None, outputs=[trends_info, trends_html])

    def get_logs_json():
        return json.dumps(ANALYTICS_LOGS[-200:], indent=2)

    demo.load(get_logs_json, inputs=None, outputs=[logs_json])

    def clear_logs():
        global ANALYTICS_LOGS
        ANALYTICS_LOGS = []
        return "Cleared logs." , get_logs_json()

    clear_logs_btn.click(clear_logs, inputs=None, outputs=[gr.Markdown(), logs_json])

    # Stylist chat
    def on_chat_send(text: str):
        if not text:
            return "Say something to the stylist."
        res = orchestrator.handle_message(DEMO_USER_ID, text)
        if res.get('type') == 'outfit':
            outfit = res['outfit']
            items = [product_by_id(pid).__dict__ for pid in outfit.get('item_ids', []) if product_by_id(pid)]
            md = f"**{outfit.get('name')}**\n\n{outfit.get('description')}\n\nTotal: ${outfit.get('estimated_price',0):.2f}\n\nWhy: A balanced look with good color harmony."
            return md
        return "I can help style outfits."

    chat_send.click(on_chat_send, inputs=[chat_input], outputs=[chat_out])

    # Quick match from chat panel
    def quick_match_click(top_choice):
        matches = recommend_matches(top_choice, DEMO_USER_ID, limit=3)
        return build_match_cards_html(matches)

    quick_match.click(quick_match_click, inputs=[quick_top], outputs=[gr.HTML()])

    demo.launch()

# End of file

⚠️ No GEMINI_API_KEY found. StylistAgent will use a rule-based fallback.
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://92d96ae4195cc54398.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
